# 07b — Rescue Lane Confirmatory Experiment (prereg v2.1)

라벨 개봉 전에 실행 규칙을 v2.1로 개정했습니다. v2는 보존했으며 가설, 지표, 표본, 정렬 규칙은 변경하지 않았습니다. fault1+2가 주 확증 집단이며 fault3는 별도 보고합니다.

실행 순서: source audit → label-free 표본과 tier → v2.1 저장 및 해시 → 캐시 검증과 모델 호출 → 안정성 호출 → 순위 확정 및 해시 → 라벨 개봉 → 주 분석과 민감도 분석. 실행 결과는 각 셀과 마지막 결론에 저장되어 있습니다. 완료된 결과는 고정하며 이 노트북 전체를 다시 실행해 추론하지 않습니다.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import hashlib, json, math, platform, subprocess, sys, time, re, threading
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = next((p for p in ROOT.parents if (p / "src").exists()), None)
if ROOT is None:
    raise RuntimeError("BLOCKED_SPEC_AMBIGUITY: 프로젝트 root를 찾을 수 없습니다.")
NOTEBOOK_PATH = ROOT / "notebooks" / "07b_rescue_lane_confirmatory.ipynb"
ART_DIR = ROOT / "artifacts" / "07b_rescue_lane_confirmatory"
ART_DIR.mkdir(parents=True, exist_ok=True)
START_UTC = datetime.now(timezone.utc).isoformat()
STOP_STATE = None
def _write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
def _append_log(text):
    with (ART_DIR / "trial_log.md").open("a", encoding="utf-8") as f: f.write(text.rstrip() + "\n")
def file_sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def _opaque(value, prefix):
    return hashlib.sha256(f"fabjudge-07a:{prefix}:{value}".encode("utf-8")).hexdigest()[:24]
def hard_stop(code, reason):
    global STOP_STATE
    STOP_STATE={"code":str(code),"reason":str(reason),"time_utc":datetime.now(timezone.utc).isoformat()}
    extra={"actual_API_requests":dict(globals().get("CALL_COUNTS",{}))}
    if "COST_BUDGET" in globals():
        try: extra["actual_cost_guard"]=COST_BUDGET.snapshot()
        except Exception: pass
    if "PREREG_SHA256" in globals(): extra["prereg_v2_1_bytes_sha256"]=PREREG_SHA256
    if "RANKINGS_SHA256" in globals(): extra["rankings_sha256"]=RANKINGS_SHA256
    if "ATTEMPT_LOG" in globals(): extra["api_attempt_rows"]=len(ATTEMPT_LOG)
    _append_log(f"\n## STOP\n- code: {code}\n- reason: {reason}\n- time_utc: {STOP_STATE['time_utc']}\n- progress: {json.dumps(extra,ensure_ascii=False)}\n")
    _write_json(ART_DIR/"summary.json",{"completed":False,"stop_condition":STOP_STATE,**extra})
    print(f"{code}: {reason}")
    display(Markdown(f"### 실험 중단: {code}\n\n{reason}"))
    raise RuntimeError(f"{code}: {reason}")

# Explicit user-authorized v2.1 resume; preserve the interrupted v2 record.
prior_summary_path=ART_DIR/"summary.json"
PRIOR_ATTEMPTS=[]
if prior_summary_path.exists():
    prior=json.loads(prior_summary_path.read_text(encoding="utf-8"))
    if prior.get("completed",False):
        raise RuntimeError("Completed v2.1 rankings and labels are frozen. Read saved outputs; do not rerun inference after label opening.")
    if not prior.get("completed",False) and prior.get("stop_condition"):
        if prior["stop_condition"]["code"]!="STOP_FIRST_LLM_SCHEMA_FAILURE":
            raise RuntimeError("Previous amended run stopped; further resume requires review.")
        if (ART_DIR/"eval_labels.csv").exists() or (ART_DIR/"rankings.csv").exists():
            raise RuntimeError("v2.1 amendment requires unopened labels and unfrozen ranks")
        archive=ART_DIR/"interrupted_v2"; archive.mkdir(exist_ok=True)
        for name in ("summary.json","hypothesis_tests.json","api_attempt_log_partial.json","trial_log.md"):
            source=ART_DIR/name; target=archive/name
            if source.exists() and not target.exists(): target.write_bytes(source.read_bytes())
prior_log=ART_DIR/"interrupted_v2"/"api_attempt_log_partial.json"
if prior_log.exists(): PRIOR_ATTEMPTS=json.loads(prior_log.read_text(encoding="utf-8"))
partial_inference_files=["api_attempt_log_partial.json","api_attempt_log.jsonl","jev_R2_scores.csv",
                         "llm_noRF_results.jsonl","llm_stability.csv","rankings.csv"]
if any((ART_DIR/name).exists() for name in partial_inference_files) and not prior_summary_path.exists():
    raise RuntimeError("BLOCKED_SPEC_AMBIGUITY: partial inference artifacts exist without a completed run; refusing automatic resume.")
try:
    GIT_HASH=subprocess.run(["git","rev-parse","HEAD"],cwd=ROOT,capture_output=True,text=True,check=True).stdout.strip()
except Exception: GIT_HASH=None
print("project_root:",ROOT)
print("notebook:",NOTEBOOK_PATH)
print("artifact_directory:",ART_DIR)
print("execution_start_utc:",START_UTC)
print("python/environment:",platform.python_version(),platform.platform())
print("git_commit:",GIT_HASH if GIT_HASH else "unavailable (workspace has no Git metadata)")

## 1. Source-of-truth audit

07a notebook 파일은 저장소에 없다. 실제 07a 실행 스크립트, pipeline client/schema 코드, R1/packet artifact, v1 preregistration을 직접 사용한다. 04b의 pass/reject는 저장된 구현으로 확인한다.

In [ ]:
sys.path[:0]=[str(ROOT/".venv"/"Lib"/"site-packages"),str(ROOT/"src")]
from fabjudge.huang2018 import INPUT_COLUMNS
from fabjudge.pipeline.features import causal_sensor_features,recent_lstm_features
from fabjudge.pipeline.schemas import PACKET_FIELDS,PACKET_SCHEMA,LLM_OUTPUT_SCHEMA,FORBIDDEN_PACKET_TOKENS,canonical_json,make_packet
from fabjudge.pipeline.config import JEV_MODEL_ID,LLM_MODEL_ID,RANDOM_SEED
from fabjudge.pipeline.prepare_07a import verify_preregistration
from fabjudge.pipeline import clients as clients_module
from fabjudge.pipeline.clients import JEVClient,LLMClient
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

P04B=ROOT/"notebooks"/"04b_huang2018_reproduction.ipynb"
P06A=ROOT/"notebooks"/"06a_lstm_jev_hybrid.ipynb"
P07A_NOTEBOOK=ROOT/"notebooks"/"07a_pipeline_skeleton.ipynb"
P07A_SCRIPT=ROOT/"scripts"/"run_07a.py"
P07A_PREP=ROOT/"src"/"fabjudge"/"pipeline"/"prepare_07a.py"
P07A_CLIENTS=ROOT/"src"/"fabjudge"/"pipeline"/"clients.py"
P07A_DATA=ROOT/"src"/"fabjudge"/"pipeline"/"data.py"
P07A_FEATURES=ROOT/"src"/"fabjudge"/"pipeline"/"features.py"
P07A_DIR=ROOT/"artifacts"/"07a_pipeline_skeleton"
P07A_CONFIG=P07A_DIR/"config.json"; P07A_PREREG=P07A_DIR/"prereg_07.json"
P07A_R1=P07A_DIR/"questions"/"R1.json"
P07A_PACKET_SCHEMA=P07A_DIR/"packet_schema.json"
P07A_LLM_SCHEMA=P07A_DIR/"llm_output_schema.json"
P07A_COST=P07A_DIR/"cost_estimate_07b.json"
SPLIT_PATH=ROOT/"artifacts"/"huang2018"/"split_metadata.json"
SEQUENCE_META_PATH=ROOT/"artifacts"/"huang2018"/"sequence_metadata.csv"
PRED_DIR=ROOT/"artifacts"/"huang2018"
P06A_SAMPLE_PLAN=ROOT/"artifacts"/"06a_lstm_jev"/"larger_test_01"/"sample_plan.csv"
P06A_PILOT=ROOT/"artifacts"/"06a_lstm_jev"/"larger_test_01"/"pilot_rows.csv"
P06A_SUMMARY=ROOT/"artifacts"/"06a_lstm_jev"/"larger_test_01"/"summary.json"
SOURCE_PATHS={
 "04b_notebook":P04B,"04b_source_code":ROOT/"src"/"fabjudge"/"huang2018_models.py",
 "04b_fault1_predictions":PRED_DIR/"fault1_test_predictions.parquet",
 "04b_fault2_predictions":PRED_DIR/"fault2_test_predictions.parquet",
 "04b_fault3_predictions":PRED_DIR/"fault3_test_predictions.parquet",
 "06a_notebook":P06A,"06a_80_row_sample_plan":P06A_SAMPLE_PLAN,
 "06a_80_row_artifact":P06A_PILOT,"06a_summary":P06A_SUMMARY,
 "07a_notebook":P07A_NOTEBOOK,"07a_execution_script":P07A_SCRIPT,
 "07a_prepare_code":P07A_PREP,"07a_api_client_code":P07A_CLIENTS,
 "07a_packet_builder_code":P07A_DATA,"07a_feature_code":P07A_FEATURES,
 "07a_config":P07A_CONFIG,"07a_R1_prompt":P07A_R1,
 "07a_packet_schema":P07A_PACKET_SCHEMA,"07a_LLM_output_schema":P07A_LLM_SCHEMA,
 "07a_cost_accounting":P07A_COST,"prereg_07_v1":P07A_PREREG,
}
missing=[n for n,p in SOURCE_PATHS.items() if not p.is_file() and n!="07a_notebook"]
if missing: hard_stop("BLOCKED_SPEC_AMBIGUITY",f"Required source files missing: {missing}")
print("07a_notebook_status:","found" if P07A_NOTEBOOK.exists() else "absent; executed code and frozen artifacts used")
for n,p in SOURCE_PATHS.items(): print(f"{n}: {p.resolve() if p.exists() else 'not present'}")
prereg_v1_hash=verify_preregistration(P07A_PREREG)
V1_BYTES_SHA256=file_sha256(P07A_PREREG)
R1=json.loads(P07A_R1.read_text(encoding="utf-8"))
config_07a=json.loads(P07A_CONFIG.read_text(encoding="utf-8"))
packet_schema_07a=json.loads(P07A_PACKET_SCHEMA.read_text(encoding="utf-8"))
llm_schema_07a=json.loads(P07A_LLM_SCHEMA.read_text(encoding="utf-8"))
cost_07a=json.loads(P07A_COST.read_text(encoding="utf-8"))
if config_07a["jev_model_id"]!=JEV_MODEL_ID or config_07a["llm_model_id"]!=LLM_MODEL_ID:
    hard_stop("BLOCKED_SPEC_AMBIGUITY","07a artifact model IDs disagree with source configuration.")
if llm_schema_07a!=LLM_OUTPUT_SCHEMA:
    hard_stop("BLOCKED_SPEC_AMBIGUITY","07a output schema artifact differs from source-code schema.")
print("07a prereg canonical hash:",prereg_v1_hash)
print("07a prereg file-bytes SHA-256:",V1_BYTES_SHA256)
print("07a model IDs:",JEV_MODEL_ID,LLM_MODEL_ID)
print("07a generation:",config_07a["llm_generation"])

## 2. 04b reproduction, 06a regression gate, and label-free sample

테스트 모집단은 split metadata의 fault1/2/3 test sequence와 정확히 일치해야 한다. 04b RF gate를 재적용하고 06a의 기존 80행 identity 및 저장 confusion summary를 확인한다. 이 단계에서는 TTF/positive label 열을 읽지 않는다.

In [ ]:
from copy import deepcopy
FAULTS=("fault1","fault2","fault3")
FAULT_CODES={"fault1":1,"fault2":2,"fault3":3}
ALARM_LIMIT_SECONDS=5000
RF_PASS_THRESHOLD=.5
PERCENTILES=[5,15,25,35,45,55,65,75,85,95]
SPLITS=json.loads(SPLIT_PATH.read_text(encoding="utf-8"))["splits"]
SEQ_META=pd.read_csv(SEQUENCE_META_PATH,usecols=["sequence_id","sequence_path"]).set_index("sequence_id")
SAFE=["sequence_id","raw_row_index","lstm_pred_seconds","rf_probability","pipeline_rul_seconds"]
PREDICTIONS={}
for fault in FAULTS:
    expected=set(SPLITS[fault]["test_original_sequences"])
    dev=set(SPLITS[fault]["train_original_sequences"])|set(SPLITS[fault]["validation_original_sequences"])
    if expected&dev: hard_stop("BLOCKED_SPEC_AMBIGUITY",f"{fault}: test and dev sequence overlap.")
    pred=pd.read_parquet(PRED_DIR/f"{fault}_test_predictions.parquet",columns=SAFE)
    pred["sequence_id"]=pred.sequence_id.astype(str)
    if set(pred.sequence_id.unique())!=expected or pred.duplicated(["sequence_id","raw_row_index"]).any():
        hard_stop("STOP_04B_REPRODUCTION_FAILED",f"{fault}: test IDs or row identities differ.")
    if not np.isfinite(pred.lstm_pred_seconds).all() or not np.isfinite(pred.rf_probability).all():
        hard_stop("STOP_04B_REPRODUCTION_FAILED",f"{fault}: nonfinite gate input.")
    if not np.array_equal(pred.rf_probability.to_numpy(float)>=.5,pred.pipeline_rul_seconds.notna().to_numpy()):
        hard_stop("STOP_04B_REPRODUCTION_FAILED",f"{fault}: RF>=.5 fails to reproduce 04b output.")
    PREDICTIONS[fault]=pred.sort_values(["sequence_id","raw_row_index"],kind="stable").reset_index(drop=True)

plan=pd.read_csv(P06A_SAMPLE_PLAN,usecols=["sequence_id","raw_row_index","sample_position"])
pilot_safe=pd.read_csv(P06A_PILOT,usecols=["sequence_id","raw_row_index","lstm_pred_seconds","rf_probability","pipeline_alarm"])
keys=["sequence_id","raw_row_index"]
if len(plan)!=80 or plan.duplicated(keys).any() or plan.sequence_id.nunique()!=8:
    hard_stop("STOP_04B_REPRODUCTION_FAILED","06a does not provide 80 unique rows from 8 sequences.")
if not plan[keys].sort_values(keys).reset_index(drop=True).equals(pilot_safe[keys].sort_values(keys).reset_index(drop=True)):
    hard_stop("STOP_04B_REPRODUCTION_FAILED","06a sample plan and pilot identities differ.")
if not set(map(tuple,plan[keys].to_numpy())).issubset(set(map(tuple,PREDICTIONS["fault3"][keys].to_numpy()))):
    hard_stop("STOP_04B_REPRODUCTION_FAILED","A 06a row is absent from fault3 04b predictions.")
reapplied_06a=(pilot_safe.lstm_pred_seconds.to_numpy(float)<=5000)&(pilot_safe.rf_probability.to_numpy(float)>=.5)
if not np.array_equal(reapplied_06a,pilot_safe.pipeline_alarm.astype(str).str.lower().eq("true").to_numpy()):
    hard_stop("STOP_04B_REPRODUCTION_FAILED","Reapplied 04b alarms differ from 06a saved alarms.")
legacy_summary=json.loads(P06A_SUMMARY.read_text(encoding="utf-8"))
legacy=next((r for r in legacy_summary["metrics"] if r["system"]=="RF to LSTM pipeline"),None)
if not legacy or tuple(legacy[k] for k in ("TP","FP","FN"))!=(9,4,7):
    hard_stop("STOP_04B_REPRODUCTION_FAILED","Preserved 06a reference does not show TP=9, FP=4, FN=7.")
REPRO80_KEYS=plan[keys].copy()
REPRO80_EXPECTED={"TP":9,"FP":4,"FN":7,"TN":int(legacy["TN"])}
print("04b rule: LSTM alarm AND RF probability >= .5")
print("06a exact 80-row identities matched; saved confusion:",REPRO80_EXPECTED)

# Percentiles are selected only within test sequence LSTM-alarm endpoints.
from collections import defaultdict
SAMPLE_KEYS_LIST=[]; SEQUENCE_SAMPLE_REPORT=[]
for fault in FAULTS:
    pred=PREDICTIONS[fault]
    for sid in sorted(set(SPLITS[fault]["test_original_sequences"])):
        alarms=pred.loc[pred.sequence_id.eq(sid)&pred.lstm_pred_seconds.le(5000)].sort_values("raw_row_index",kind="stable").reset_index(drop=True)
        n=len(alarms); pos=defaultdict(list)
        for pct in PERCENTILES:
            if n: pos[math.floor(pct/100*(n-1))].append(pct)
        SEQUENCE_SAMPLE_REPORT.append({"fault":fault,"sequence":sid,"alarm_endpoint_rows":n,
            "requested_percentile_positions":len(PERCENTILES) if n else 0,"unique_selected_rows":len(pos)})
        for i,pcts in pos.items():
            SAMPLE_KEYS_LIST.append({"fault":fault,"sequence_id":sid,
                "raw_row_position":int(alarms.iloc[i].raw_row_index),"sampled_percentile":";".join(map(str,pcts))})
SAMPLE_KEYS=pd.DataFrame(SAMPLE_KEYS_LIST)
if SAMPLE_KEYS.empty or SAMPLE_KEYS.duplicated(["sequence_id","raw_row_position"]).any():
    hard_stop("BLOCKED_SPEC_AMBIGUITY","Percentile sampling is empty or duplicated.")

# 06e feature code is reused with 04b test prediction histories; sequence NPZ x/raw positions only.
ARRAY_CACHE={}; SAMPLE_RECORDS=[]
for fault in FAULTS:
    pred=PREDICTIONS[fault]
    joined=SAMPLE_KEYS.loc[SAMPLE_KEYS.fault.eq(fault)].merge(
        pred,left_on=["sequence_id","raw_row_position"],right_on=["sequence_id","raw_row_index"],
        how="left",validate="one_to_one")
    for item in joined.to_dict(orient="records"):
        sid=str(item["sequence_id"])
        if sid not in ARRAY_CACHE:
            rel=Path(str(SEQ_META.loc[sid,"sequence_path"]).replace("\\","/"))
            path=(ROOT/rel).resolve()
            seq_root=(ROOT/"artifacts"/"huang2018"/"sequences").resolve()
            if not path.is_relative_to(seq_root): hard_stop("BLOCKED_SPEC_AMBIGUITY","Sequence path escaped 04b artifact directory.")
            with np.load(path,allow_pickle=False) as arrays: ARRAY_CACHE[sid]=(arrays["x"],arrays["raw_row_index"])
        x,raw_index=ARRAY_CACHE[sid]; rawpos=int(item["raw_row_position"])
        sensorpos=int(np.searchsorted(raw_index,rawpos))
        if sensorpos>=len(raw_index) or int(raw_index[sensorpos])!=rawpos:
            hard_stop("BLOCKED_SPEC_AMBIGUITY",f"Sensor row missing: {sid}/{rawpos}.")
        hist=pred.loc[pred.sequence_id.eq(sid)].sort_values("raw_row_index",kind="stable").reset_index(drop=True)
        predpos=int(np.searchsorted(hist.raw_row_index.to_numpy(),rawpos))
        recent=recent_lstm_features(hist.lstm_pred_seconds.to_numpy(float),predpos)
        sensors=causal_sensor_features(x,sensorpos,INPUT_COLUMNS)
        rf=float(item["rf_probability"]); fields={name:None for name in PACKET_FIELDS}
        fields.update({"lstm_pred_seconds":float(item["lstm_pred_seconds"]),"rf_probability":rf,"fault_type":FAULT_CODES[fault]})
        fields.update(recent); fields.update(sensors)
        packet=make_packet(fields); packet["packet_version"]="07a-evidence-v1-noRF"; packet["fields"].pop("rf_probability")
        rowkey=hashlib.sha256(f"07b:{fault}:{sid}:{rawpos}".encode()).hexdigest()[:32]
        pct=int(str(item["sampled_percentile"]).split(";")[0])
        SAMPLE_RECORDS.append({
            "row_key":rowkey,"row_id":_opaque(f"{sid}:{rawpos}:{pct}","row"),
            "sequence_group":_opaque(sid,"sequence-group"),"fault":fault,"sequence_id":sid,
            "sampled_percentile":item["sampled_percentile"],"original_row_position":rawpos,
            "lstm_pred_seconds":float(item["lstm_pred_seconds"]),"rf_probability":rf,"lstm_alarm":True,
            "04b_pass":rf>=.5,"04b_decision":"pass" if rf>=.5 else "reject",
            "tier":"tier1" if rf>=.5 else "rescue","packet_json":canonical_json(packet),
        })
SAMPLE=pd.DataFrame(SAMPLE_RECORDS)
def packet_key_guard(packet):
    if set(packet)!={"packet_version","fields"} or packet["packet_version"]!="07a-evidence-v1-noRF": return False
    if set(packet["fields"])!=set(PACKET_FIELDS)-{"rf_probability"}: return False
    def walk(x):
        if isinstance(x,dict):
            for k,v in x.items():
                low=str(k).lower()
                if "rf" in low or any(t in low for t in FORBIDDEN_PACKET_TOKENS) or not walk(v): return False
        elif isinstance(x,list): return all(walk(v) for v in x)
        return True
    return walk(packet)
if SAMPLE.empty or SAMPLE.row_key.duplicated().any(): hard_stop("BLOCKED_SPEC_AMBIGUITY","Sample empty or row keys collide.")
for s in SAMPLE.packet_json:
    if not packet_key_guard(json.loads(s)): hard_stop("STOP_LABEL_OR_IDENTIFIER_LEAK","Packet contains identifier, label, RF, or unexpected field.")
if SAMPLE.loc[SAMPLE.tier.eq("rescue"),"packet_json"].map(lambda s:all(v is not None for v in json.loads(s)["fields"].values())).eq(False).any():
    hard_stop("BLOCKED_SPEC_AMBIGUITY","Rescue packet is DATA_REVIEW under existing no-imputation rule.")

tier_cols=["row_key","fault","sequence_id","sampled_percentile","original_row_position","lstm_pred_seconds","lstm_alarm","rf_probability","04b_decision","tier"]
SAMPLE[tier_cols].to_csv(ART_DIR/"tier_split.csv",index=False,encoding="utf-8")
SAMPLE[["row_key","row_id","sequence_group","fault","sequence_id","sampled_percentile",
 "original_row_position","lstm_pred_seconds","rf_probability","lstm_alarm","04b_pass","tier","packet_json"]].to_csv(
 ART_DIR/"eval_sample.csv",index=False,encoding="utf-8")
sequence_report=pd.DataFrame(SEQUENCE_SAMPLE_REPORT)
display(sequence_report)
tier_counts=pd.DataFrame([{
 "fault":f,"tier1_rows":int(((SAMPLE.fault==f)&(SAMPLE.tier=="tier1")).sum()),
 "rescue_rows":int(((SAMPLE.fault==f)&(SAMPLE.tier=="rescue")).sum()),
 "tier1_sequences":SAMPLE.loc[(SAMPLE.fault==f)&(SAMPLE.tier=="tier1"),"sequence_id"].nunique(),
 "rescue_sequences":SAMPLE.loc[(SAMPLE.fault==f)&(SAMPLE.tier=="rescue"),"sequence_id"].nunique(),
} for f in FAULTS])
display(tier_counts)
print("label-free evaluation rows:",len(SAMPLE))

## 3. Preregistration v2.1 and cumulative budget

v2 원본을 보존하고 v2.1에 개정 이유, 같은 요청의 파싱/스키마 재시도 2회, 최초 20건 중 실패 3건 이상 중단, 전체 rescue LLM 실패율 5% 초과 시 기술 통계만 보고하는 규칙을 기록했습니다. 기존 사용액을 포함해 v2의 비용 상한을 유지합니다. 모델 요청 설정은 동일하며 캐시는 정확한 요청 해시가 일치할 때만 재사용합니다.


In [ ]:
cost_n=cost_07a.get("jev_cost_sample_size"); cost_sum=cost_07a.get("observed_jev_cost_usd")
llm_unit=cost_07a.get("mean_smoke_cost_usd")
if not isinstance(cost_n,int) or cost_n<=0 or not isinstance(cost_sum,(int,float)):
    hard_stop("BLOCKED_SPEC_AMBIGUITY","07a JEV observed cost unit is unavailable.")
if not isinstance(llm_unit,(int,float)) or llm_unit<=0:
    hard_stop("BLOCKED_SPEC_AMBIGUITY","07a LLM smoke mean cost is unavailable.")
jev_unit=float(cost_sum)/cost_n
RESCUE_N=int(SAMPLE.tier.eq("rescue").sum())
stab_frame=SAMPLE.loc[SAMPLE.tier.eq("rescue")].sort_values(["sequence_group","row_id"],kind="stable").reset_index(drop=True)
stab_n=min(5,len(stab_frame)); stab_rng=np.random.default_rng(RANDOM_SEED)
STABILITY_KEYS=stab_frame.iloc[np.sort(stab_rng.choice(len(stab_frame),size=stab_n,replace=False))].row_key.tolist()
EXPECTED_JEV=RESCUE_N; EXPECTED_MAIN_LLM=RESCUE_N; EXPECTED_STABILITY=stab_n*3
EXPECTED_TOTAL=EXPECTED_JEV+EXPECTED_MAIN_LLM+EXPECTED_STABILITY
EST_JEV=EXPECTED_JEV*jev_unit
EST_LLM=(EXPECTED_MAIN_LLM+EXPECTED_STABILITY)*float(llm_unit)
EST_TOTAL=EST_JEV+EST_LLM
COST_CAP=EST_TOTAL*1.30
seq_counts={}
for f in FAULTS:
    s=sequence_report.loc[sequence_report.fault.eq(f)]
    seq_counts[f]={
        "test_sequences":int(s.sequence.nunique()),
        "alarm_endpoint_rows":int(s.alarm_endpoint_rows.sum()),
        "requested_percentile_positions":int(s.requested_percentile_positions.sum()),
        "unique_sampled_rows":int(SAMPLE.fault.eq(f).sum()),
    }
R2_PREREG=deepcopy(R1)
_r2_rf_sentence="rf_probability: sequence-group out-of-fold estimated probability of a fault within 5000 seconds. "
if _r2_rf_sentence not in R2_PREREG["instructions"]: raise RuntimeError("BLOCKED_SPEC_AMBIGUITY: exact RF sentence absent from R1")
R2_PREREG["instructions"]=R2_PREREG["instructions"].replace(_r2_rf_sentence,"",1)
R2_PREREG["criteria"]=[c.replace(", RF signal","") for c in R2_PREREG["criteria"]]
revision_reason=[
 "v1 표본에서는 fault1/2 양성이 0건이어서 주 지표를 정의할 수 없었다.",
 "07c를 제거하고 07b를 최종 confirmatory experiment로 전환한다.",
 "최종 비교 대상을 04b로 명시하고 structural rescue lane 설계를 채택한다.",
 "07a에서는 JEV/LLM 성과를 ground-truth label로 평가한 적이 없다.",
]
PREREG={
 "protocol":"FabJudge 07b rescue lane confirmatory experiment","version":"prereg_07_v2",
 "revision_reason":revision_reason,
 "source_of_truth_paths":{k:str(v.resolve()) for k,v in SOURCE_PATHS.items()},
 "preserved_v1":{"path":str(P07A_PREREG.resolve()),"file_bytes_sha256":V1_BYTES_SHA256},
 "sampling":{
  "population":"fault1/fault2/fault3 test original sequences only, from split_metadata.json.",
  "endpoint_source":"04b fault-specific test prediction parquet; no label columns loaded before ranking freeze.",
  "alarm":"lstm_pred_seconds <= 5000 seconds.","percentiles":PERCENTILES,
  "index_rule":"floor((percentile/100)*(n_alarm_endpoints-1))",
  "duplicate_index":"Coalesce identical row positions and record all percentiles mapped to each row.",
  "row_cap":None,"counts_by_fault":seq_counts,"total_unique_rows":len(SAMPLE),"audit_sampling":False,
 },
 "tier":{
  "04b_rule":"Saved Huang rule: RF probability >= 0.5; gated LSTM output must agree.",
  "tier1":"LSTM alarm AND 04b pass; immediate review.",
  "rescue":"LSTM alarm AND 04b reject; structural rescue population.",
  "06a_regression":{"rows":80,"expected_confusion":REPRO80_EXPECTED},
 },
 "JEV_R2":{
  "source":str(P07A_R1.resolve()),
  "definition":"07a R1 with only the RF probability sentence and RF references in criteria removed.",
  "question":R2_PREREG,"question_sha256":hashlib.sha256(canonical_json(R2_PREREG).encode("utf-8")).hexdigest(),
  "population":"All rescue rows.","score":"07a native score [0,2], no numeric imputation.",
  "unavailable":"LOW after scored LOW rows.",
 },
 "LLM_packet":{
  "definition":"07a packet v1 with RF field removed; packet_version 07a-evidence-v1-noRF.",
  "packet_version":"07a-evidence-v1-noRF","field_order":[f for f in PACKET_FIELDS if f!="rf_probability"],
  "fields":sorted(set(PACKET_FIELDS)-{"rf_probability"}),
  "model_id":config_07a["llm_model_id"],"generation":config_07a["llm_generation"],
  "output_schema_sha256":file_sha256(P07A_LLM_SCHEMA),
  "population":"Call every rescue row once; reuse the output in every routing comparison.",
 },
 "stability":{
  "selection":"07a seeded random choice after sorting sequence_group,row_id; numpy default_rng(20260927), without replacement.",
  "seed":RANDOM_SEED,"selected_row_keys":STABILITY_KEYS,"rows":stab_n,
  "calls_per_row":3,"additional_calls":EXPECTED_STABILITY,
 },
 "routing":{
  "JEV":{
   "population":"Within each fault's valid-score rescue rows.",
   "segments":{"HIGH":"top 25%","MID":"middle 50%","LOW":"bottom 25%"},
   "rounding":"round-half-up; score descending then internal row key ascending.",
   "queue":"HIGH then MID then LOW.",
   "HIGH_order":"JEV score descending.",
   "MID_order":"LLM priority_score descending; tie JEV score descending then row key.",
   "LOW_order":"JEV score descending; unavailable JEV last.",
   "unavailable":"LOW after valid LOW; MID is half-up 50% of valid JEV scores.",
   "MID_uses":"The single existing full-rescue LLM output; no route-specific calls.",
   "audit_sample":False,
  },
  "full_LLM":"priority_score descending; MODEL_UNAVAILABLE last; no imputation; row-key tie.",
  "random":{
   "LLM_budget":"Exact integer JEV MID count per fault.",
   "remaining":"Random HIGH/LOW allocation in half-up equal counts; random order.",
   "MID_order":"LLM priority_score descending; randomized tie order.",
   "repetitions":1000,"seed":RANDOM_SEED+1,
  },
  "LSTM":{
   "score":"-lstm_pred_seconds","segments":{"HIGH":"top 25%","MID":"middle 50%","LOW":"bottom 25%"},
   "rounding":"round-half-up; score descending then row key.",
   "MID_order":"LLM priority_score descending; tie by -lstm_pred_seconds.",
  },
  "no_LLM_baselines":{
   "JEV_only":"JEV score descending.","LSTM_only":"-lstm_pred_seconds descending.",
   "Logistic":"07a v1-noRF numeric fields; READY train+validation only; StandardScaler; LogisticRegression(C=1,solver=lbfgs,max_iter=5000).",
  },
  "same_LLM_failure":"Reuse the same full-rescue output/unavailable status in every routing comparison.",
 },
 "review_budgets":{"E":[0.10,0.20,0.30],"primary_E":0.20,
  "rounding":"round-half-up per fault rescue count.","combined":"Pool each fault-specific top-E reviewed rows and positives."},
 "label":{"positive":"TTF <= 5000 sec","unit":"seconds"},
 "primary_population":"fault1+fault2 combined rescue; fault3 separately reported.",
 "feasibility":{"population":"fault1+fault2 combined rescue","minimum_positive_rows":30,
  "minimum_negative_rows":30,"minimum_rescue_sequences":10,
  "failure":"INSUFFICIENT_CONFIRMATORY_SAMPLE; no H1-H3 formal conclusion, descriptive/exploratory continue."},
 "hypotheses":{
  "order":["H1","H2","H3"],"alpha":0.05,"bootstrap_repetitions":10000,
  "bootstrap":{"unit":"Sequence resampling with replacement within fault; keep all rescue rows.",
   "seed":RANDOM_SEED,"lower_bound":"5th percentile; one-sided 95%.",
   "H3_randomization":"Point comparator is mean of 1,000 seeded budget-matched routes; each sequence-bootstrap replicate uses exact conditional expected precision under the same randomized HIGH/MID/LOW assignment, preserving the frozen LLM order within MID.",
   "E":"Recompute half-up E20 per fault for each bootstrap sample and use frozen rank."},
  "H1":"JEV precision@E20 minus rescue positive prevalence; lower bound > 0.",
  "H2":"JEV precision@E20 minus full LLM precision@E20; margin -0.05; lower bound > -0.05.",
  "H3":"JEV precision@E20 minus mean of 1,000 random-routing precision@E20; lower bound > 0.",
  "gatekeeping":"Test H2 only after H1 passes; test H3 only after H1 and H2 pass.",
 },
 "exploratory_only":["JEV vs LSTM routing","full LLM vs logistic","full LLM vs LSTM-only",
  "E10","E30","fault1","fault2","fault3","stability","JEV-only","LSTM-only"],
 "effect_vs_04b":{
  "rescue_recovery":"top-E rescue positives / all sampled rescue positives.",
  "04b_recall":"tier1 positives / all sampled positives.",
  "structural_recall":"(tier1 positives + top-E rescue positives) / all sampled positives.",
 },
 "cost_curve":{"review_cost":1,"missed_failure_cost_r":list(range(1,21)),
  "total":"review_count*1 + missed_positive_count*r",
  "break_even":"additional review rows / additional recovered positives when positive count is nonzero."},
 "call_budget":{
  "max_concurrency":4,"retry":{"429_or_5xx":3,"backoff_seconds":[1,2,4]},
  "expected_calls":{"JEV":EXPECTED_JEV,"main_LLM":EXPECTED_MAIN_LLM,
   "stability_LLM":EXPECTED_STABILITY,"total":EXPECTED_TOTAL},
  "cost_source":"07a cost_estimate_07b: observed JEV mean call cost; LLM smoke mean cost.",
  "estimated_cost_usd":{"JEV":EST_JEV,"LLM":EST_LLM,"total":EST_TOTAL},"cap_usd":COST_CAP,
  "rule":"Stop before next request if known actual cumulative cost plus preregistered unit estimate exceeds cap.",
 },
 "failure_handling":{
  "exhausted_retry":"MODEL_UNAVAILABLE; keep numeric output missing.",
  "first_schema_failure":"Stop with STOP_FIRST_JEV_SCHEMA_FAILURE or STOP_FIRST_LLM_SCHEMA_FAILURE.",
  "LLM_unavailable_ranking":"Sort last; use same status for all routing comparisons.",
  "no_route_specific_calls":True,
 },
 "stop_conditions":["STOP_04B_REPRODUCTION_FAILED","STOP_LABEL_OR_IDENTIFIER_LEAK",
  "STOP_FIRST_JEV_SCHEMA_FAILURE","STOP_FIRST_LLM_SCHEMA_FAILURE","STOP_CALL_BUDGET_EXCEEDED",
  "STOP_PREREG_HASH_MISMATCH","BLOCKED_SPEC_AMBIGUITY"],
 "test_history_note":"fault3 test was examined in 06 series and is not completely unseen.",
}
V2_PATH=ART_DIR/"prereg_07_v2.json"
V2_SHA256=file_sha256(V2_PATH)
assert V2_SHA256=="9eb9fe21d46451d1bc022c75d28436460d6b411b425e005a6bd8e14d31524cbe"
PREREG=json.loads(V2_PATH.read_text(encoding="utf-8"))
PREREG["version"]="prereg_07_v2_1"
PREREG["preserved_v2"]={"path":str(V2_PATH),"file_bytes_sha256":V2_SHA256}
PREREG["amendment_reason"]="첫 LLM 응답 1건이 JSON 뒤 추가 텍스트로 파싱에 실패해 STOP_FIRST_LLM_SCHEMA_FAILURE로 중단됨.\n라벨 미개봉, 순위 미확정 상태에서 실행 규칙만 개정함. 가설, 지표, 표본, 정렬 규칙은 변경 없음."
PREREG["amendment"]={
 "label_state":"unopened", "ranking_state":"not frozen",
 "schema_retry":"Same request body, at most 2 additional attempts for LLM parsing/schema failure; complete json.loads only; never partial JSON.",
 "first20":"First 20 rescue rows sorted by row_key, including same-request successful caches; stop if at least 3 invalid after retries.",
 "unavailable_rate":"Main rescue LLM unavailable rows / all rescue rows > 0.05: descriptive statistics only, no formal hypothesis testing.",
 "diagnostics":"Save failed raw text under failed_raw with secrets redacted. Print one actual request body with strict json_schema.",
 "response_format":{"already_present":True,"settings_changed":False,"type":"json_schema","strict":True},
 "cache":"Reuse only exact canonical request SHA-256 match; same settings as v2.",
 "stability_implementation":"Restrict calls to the five already preregistered STABILITY_KEYS, 3 repeats each.",
 "sensitivity":"Exclude main LLM MODEL_UNAVAILABLE rows from every frozen ranking, preserve relative order, recompute per-fault E20 and H1-H3 with same sequence bootstrap and gatekeeping; sensitivity only, primary decisions unchanged.",
 "historical_failed_raw":"Original v2 client did not persist the failed raw response; it cannot be recovered. New failures are persisted.",
}
PREREG["call_budget"]["retry"]["LLM_parse_or_schema"]=2
PREREG["failure_handling"]["first_schema_failure"]="JEV: STOP_FIRST_JEV_SCHEMA_FAILURE; LLM: retry twice, MODEL_UNAVAILABLE on exhaustion."
PREREG["failure_handling"]["first20_LLM"]="Stop if >=3 invalid after retries among first 20 main rows."
PREREG["failure_handling"]["LLM_unavailable_rate"]="More than 5% main rescue rows unavailable: descriptive only."
PREREG["stop_conditions"]=[x for x in PREREG["stop_conditions"] if x!="STOP_FIRST_LLM_SCHEMA_FAILURE"]+["STOP_FIRST20_LLM_SCHEMA_FAILURES"]
PREREG_PATH=ART_DIR/"prereg_07_v2_1.json"
PREREG_BYTES=(json.dumps(PREREG,ensure_ascii=False,indent=2,allow_nan=False)+"\n").encode("utf-8")
if PREREG_PATH.exists() and PREREG_PATH.read_bytes()!=PREREG_BYTES:
    hard_stop("STOP_PREREG_HASH_MISMATCH","Existing v2.1 differs; refusing overwrite")
if not PREREG_PATH.exists(): PREREG_PATH.write_bytes(PREREG_BYTES)
PREREG_SHA256=file_sha256(PREREG_PATH)
def assert_prereg_hash():
    if file_sha256(PREREG_PATH)!=PREREG_SHA256 or file_sha256(V2_PATH)!=V2_SHA256:
        hard_stop("STOP_PREREG_HASH_MISMATCH","Preregistration bytes changed")
    return PREREG_SHA256
_append_log(f"\n## v2.1 amendment before labels\n- {PREREG['amendment_reason']}\n- v2 SHA-256: {V2_SHA256}\n- v2.1 SHA-256: {PREREG_SHA256}\n- amendment UTC: {START_UTC}\n- strict json_schema already present; request settings unchanged.\n- Original failed raw unavailable (not persisted by v2 client).\n")
assert_prereg_hash()
print("prereg_07_v2_1 SHA-256:",PREREG_SHA256)
print("expected logical calls and cumulative cap:",EXPECTED_TOTAL,COST_CAP)


## 4. JEV R2 diff, leakage/order guard, and exact model check

R2는 R1의 RF 전용 문장과 criteria 속 RF 참조만 삭제한다. output schema/generation은 07a와 동일하다. inference 전에 packet field, notebook cell order, credential roles 및 exact model catalog를 확인한다.

In [ ]:
def build_r2(r1):
    q=deepcopy(r1)
    rf_sentence="rf_probability: sequence-group out-of-fold estimated probability of a fault within 5000 seconds. "
    if rf_sentence not in q["instructions"]: hard_stop("BLOCKED_SPEC_AMBIGUITY","Exact RF sentence was not found in R1.")
    q["instructions"]=q["instructions"].replace(rf_sentence,"",1)
    q["criteria"]=[c.replace(", RF signal","") for c in q["criteria"]]
    return q
R2=build_r2(R1)
if R2!=build_r2(R1) or "rf" in (R2["instructions"]+" ".join(R2["criteria"])).lower():
    hard_stop("BLOCKED_SPEC_AMBIGUITY","R2 contains changes beyond RF removal.")
import difflib
print("\n".join(difflib.unified_diff(json.dumps(R1,ensure_ascii=False,indent=2).splitlines(),
 json.dumps(R2,ensure_ascii=False,indent=2).splitlines(),fromfile="07a R1",tofile="07b R2",lineterm="")))
(ART_DIR/"questions").mkdir(parents=True,exist_ok=True)
_write_json(ART_DIR/"questions"/"R2.json",R2)
PACKET_SCHEMA_NORF=deepcopy(PACKET_SCHEMA)
PACKET_SCHEMA_NORF["$id"]="https://fabjudge.local/schemas/evidence-packet-07a-v1-noRF.json"
PACKET_SCHEMA_NORF["title"]="FabJudge EvidencePacket 07a v1-noRF"
PACKET_SCHEMA_NORF["properties"]["packet_version"]={"const":"07a-evidence-v1-noRF"}
PACKET_SCHEMA_NORF["properties"]["fields"]["required"]=[x for x in PACKET_SCHEMA_NORF["properties"]["fields"]["required"] if x!="rf_probability"]
PACKET_SCHEMA_NORF["properties"]["fields"]["properties"].pop("rf_probability")
PACKET_SCHEMA_NORF["x-field-metadata"].pop("rf_probability")
def validate_packet_no_rf(packet):
    if not packet_key_guard(packet): raise ValueError("Packet v1-noRF has invalid keys.")
    from jsonschema import Draft202012Validator
    Draft202012Validator(PACKET_SCHEMA_NORF).validate(packet)
clients_module.validate_packet=validate_packet_no_rf

nb=nbformat.read(NOTEBOOK_PATH,as_version=4)
# STATIC_ORDER_GUARD
freeze_cells=[i for i,c in enumerate(nb.cells) if c.cell_type=="code" and any(line.strip()=="# RANKINGS_FINALIZATION" for line in c.source.splitlines())]
label_cells=[i for i,c in enumerate(nb.cells) if c.cell_type=="code" and any(line.strip()=="# LABELS_LOAD" for line in c.source.splitlines())]
guard_cells=[i for i,c in enumerate(nb.cells) if c.cell_type=="code" and any(line.strip()=="# STATIC_ORDER_GUARD" for line in c.source.splitlines())]
if len(freeze_cells)!=1 or len(label_cells)!=1 or len(guard_cells)!=1 or freeze_cells[0]>=label_cells[0]:
    hard_stop("STOP_LABEL_OR_IDENTIFIER_LEAK","Static cell order puts labels before ranking freeze.")
if any(c.cell_type=="code" and re.search(r"\bload_eval_labels\s*\(\s*\)",c.source) and i<label_cells[0] and i not in guard_cells for i,c in enumerate(nb.cells)):
    hard_stop("STOP_LABEL_OR_IDENTIFIER_LEAK","A label loader call occurs before LABELS_LOAD.")
if config_07a["llm_generation"]!={"temperature":0,"reasoning_effort":"none","max_tokens":700,"response_format":"strict_json_schema"}:
    hard_stop("BLOCKED_SPEC_AMBIGUITY","07a generation settings differ.")
assert_prereg_hash()
try:
    from fabjudge.pipeline.keys import ensure_distinct_keys,load_jev_key,load_llm_key
    ensure_distinct_keys(ROOT); load_jev_key(ROOT); load_llm_key(ROOT)
    cat=LLMClient(ROOT,ART_DIR/"cache"/"catalog_only",max_new_calls=0)
    MODEL_METADATA=cat.catalog_model()
except Exception as exc:
    hard_stop("MODEL_UNAVAILABLE",f"Exact 07a model/credential/catalog check failed: {exc}")
if MODEL_METADATA.get("model_id")!=config_07a["llm_model_id"]:
    hard_stop("BLOCKED_SPEC_AMBIGUITY","Catalog does not verify exact 07a model ID.")
_write_json(ART_DIR/"model_catalog_verification.json",MODEL_METADATA)
print("R1/R2 unified diff checked; packet v1-noRF field count:",len(PACKET_FIELDS)-1)
print("model ID:",MODEL_METADATA["model_id"],"generation:",config_07a["llm_generation"])
print("label ordering guard PASS; inference has not started.")

## 5. Label-blind JEV R2 and full-rescue LLM calls

JEV R2는 rescue 전체에 적용합니다. LLM 결과 하나를 모든 비교 방식에서 공유합니다. JSON 전체 파싱과 스키마 검증 실패는 동일 요청으로 최대 2회 재시도하며, 끝내 실패하면 MODEL_UNAVAILABLE로 유지합니다. 실패 원문은 비밀키를 가린 뒤 failed_raw에 저장합니다. 실제 제출 요청 body 한 건에서 strict json_schema를 출력합니다.

row_key 순서의 첫 20행(동일 요청 성공 캐시 포함)에서 재시도 후 유효하지 않은 응답이 3건 이상이면 중단합니다. 전체 main rescue LLM의 MODEL_UNAVAILABLE 비율이 5%를 초과하면 공식 가설 검정을 하지 않습니다. JEV 최초 스키마 실패 중단 규칙은 유지합니다.


In [ ]:
from concurrent.futures import ThreadPoolExecutor,as_completed
from threading import Lock,Event

class BudgetExceeded(RuntimeError): pass
class CostBudget:
    def __init__(self,cap):
        self.cap=float(cap); self.confirmed=0.0; self.reserved=0.0
        self.reported=0.0; self.fallback=0.0; self.lock=Lock()
    def reserve(self,unit):
        with self.lock:
            if self.confirmed+self.reserved+unit>self.cap+1e-12:
                raise BudgetExceeded(f"confirmed+reserved+next_unit exceeds ${self.cap:.8f}")
            self.reserved+=unit
    def settle(self,unit,was_api_call,reported_cost):
        with self.lock:
            self.reserved=max(0.0,self.reserved-unit)
            if not was_api_call: return
            if isinstance(reported_cost,(int,float)) and math.isfinite(float(reported_cost)) and reported_cost>=0:
                measured=float(reported_cost); self.reported+=measured
            else:
                measured=float(unit); self.fallback+=measured
            self.confirmed+=measured
    def snapshot(self):
        with self.lock: return {"guard_spend_usd":self.confirmed,"reported_cost_usd":self.reported,
                                "fallback_estimated_cost_usd":self.fallback,"reserved_usd":self.reserved,
                                "cap_usd":self.cap}
COST_BUDGET=CostBudget(COST_CAP)
ABORT=Event(); ATTEMPT_LOG=list(PRIOR_ATTEMPTS)
for a in PRIOR_ATTEMPTS:
    if not a.get("cache_hit",False):
        unit=jev_unit if a.get("call_stage")=="jev" else float(llm_unit)
        COST_BUDGET.reserve(unit); COST_BUDGET.settle(unit,True,a.get("cost_usd"))
CALL_COUNTS={"jev":0,"llm_main":0,"llm_stability":0}
for a in PRIOR_ATTEMPTS:
    if not a.get("cache_hit",False): CALL_COUNTS[a["call_stage"]]+=1
CALL_LOCK=Lock()
FAILED_DIR=ART_DIR/"failed_raw"; FAILED_DIR.mkdir(exist_ok=True)
DIAG_LOCK=Lock(); REQUEST_PRINTED=False
class DiagnosticLLMClient(LLMClient):
    def __init__(self,*args,**kwargs):
        super().__init__(*args,**kwargs)
        self.last_raw=None; self.last_digest=None
        original_post=self.session.post
        def captured_post(url,*args,**kwargs):
            global REQUEST_PRINTED
            body=json.loads(kwargs["data"])
            self.last_digest=hashlib.sha256(canonical_json(body).encode("utf-8")).hexdigest()
            assert body["response_format"]["type"]=="json_schema"
            assert body["response_format"]["json_schema"]["strict"] is True
            with DIAG_LOCK:
                if not REQUEST_PRINTED:
                    _write_json(ART_DIR/"actual_request_body_example.json",body)
                    print("Actual submitted LLM request body (authorization headers omitted):")
                    print(json.dumps(body,ensure_ascii=False,indent=2))
                    REQUEST_PRINTED=True
            response=original_post(url,*args,**kwargs)
            self.last_raw=response.text
            return response
        self.session.post=captured_post
    def persist_failure(self,result):
        raw=self.last_raw
        if raw is None: return
        try: raw=json.loads(raw)["choices"][0]["message"]["content"]
        except (ValueError,KeyError,IndexError,TypeError): pass
        raw=str(raw)
        for secret in (self._key,clients_module.load_jev_key(ROOT),clients_module.load_llm_key(ROOT)):
            if secret: raw=raw.replace(secret,"[REDACTED]")
        raw=re.sub(r"sk-[A-Za-z0-9_-]{12,}","[REDACTED]",raw)
        stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
        path=FAILED_DIR/f"{self.last_digest}_{stamp}.txt"
        path.write_text(raw,encoding="utf-8")
        # Citation-invalid responses may have been cached by the base parser.
        if self.last_digest:
            cache_path=self.cache_dir/f"{self.last_digest}.json"
            if cache_path.exists(): cache_path.unlink()

JEV_CACHE=ART_DIR/"cache"/"jev_R2"; LLM_CACHE=ART_DIR/"cache"/"llm_noRF"
JEV_CACHE.mkdir(parents=True,exist_ok=True); LLM_CACHE.mkdir(parents=True,exist_ok=True)
RETRY_DELAYS=(1,2,4); MAX_RETRIES=3

def _http_status(err):
    m=re.search(r"HTTP\s+(\d{3})",str(err or ""),re.I)
    return int(m.group(1)) if m else None

def _response_schema_failure(result):
    if result.get("status")=="success": return False
    err=str(result.get("error") or "")
    if _http_status(err) is not None: return False
    transport=re.search(r"timeout|timed out|connection|urlopen|network|socket|name resolution|remote end|temporarily unavailable|broken pipe|sslerror",err,re.I)
    return not bool(transport)

def _citation_audit(output,packet):
    cited=[]
    for section in ("evidence_for","evidence_against"):
        for claim in output.get(section,[]):
            field=claim.get("field"); value=claim.get("value")
            if field not in packet["fields"] or field=="rf_probability":
                raise ValueError(f"Citation field absent from v1-noRF packet: {field}")
            packet_value=float(packet["fields"][field]); cited_value=float(value)
            if abs(cited_value-packet_value)>1e-6+1e-6*abs(packet_value):
                raise ValueError(f"Citation value mismatch for {field}")
            cited.append(field)
    return {"citation_count":len(cited),"citation_agreement":True}

def _run_one(kind,record,*,cache_dir,cache_role,repeat_tag=None):
    if ABORT.is_set(): return {"row_key":record["row_key"],"status":"CANCELLED_AFTER_STOP","attempts":[],"new_calls":0}
    unit=jev_unit if kind=="jev" else float(llm_unit)
    cls=JEVClient if kind=="jev" else DiagnosticLLMClient
    client=cls(ROOT,cache_dir,max_new_calls=6)
    packet=json.loads(record["packet_json"])
    result=None; schema_failure=False; schema_retries=0; http_retries=0
    for attempt_no in range(6):
        schema_failure=False
        if ABORT.is_set(): break
        try: COST_BUDGET.reserve(unit)
        except BudgetExceeded as exc:
            return {"row_key":record["row_key"],"status":"STOP_CALL_BUDGET_EXCEEDED","error":str(exc),
                    "attempts":client.attempts.copy(),"new_calls":client.new_calls}
        before=client.new_calls
        try:
            if kind=="jev": result=client.call(packet,R2)
            else: result=client.call(packet,LLM_OUTPUT_SCHEMA,model_metadata=MODEL_METADATA)
        except Exception as exc:
            result={"status":"MODEL_UNAVAILABLE","error":f"{type(exc).__name__}: {str(exc)[:400]}","output":None,"score":None}
        was_api=client.new_calls>before
        COST_BUDGET.settle(unit,was_api,result.get("cost_usd"))
        if was_api:
            with CALL_LOCK:
                if cache_role in CALL_COUNTS: CALL_COUNTS[cache_role]+=1
        if client.attempts:
            ATTEMPT_LOG.append({**client.attempts[-1],"row_key":record["row_key"],"call_stage":cache_role,
                                "retry_index":attempt_no,"repeat_tag":repeat_tag})
        if result.get("status")=="success":
            if kind=="llm":
                try: result.update(_citation_audit(result["output"],packet))
                except Exception as exc:
                    result={**result,"status":"MODEL_UNAVAILABLE","error":f"Citation validation: {type(exc).__name__}: {str(exc)[:300]}","output":None}
                    schema_failure=True
            if not schema_failure: break
        else:
            schema_failure=_response_schema_failure(result)
        if schema_failure:
            if kind=="jev": break
            client.persist_failure(result)
            if schema_retries>=2: break
            schema_retries+=1
            continue
        status=_http_status(result.get("error"))
        retryable=status==429 or (status is not None and 500<=status<=599)
        if not retryable or http_retries>=MAX_RETRIES: break
        time.sleep(RETRY_DELAYS[http_retries]); http_retries+=1
    if result is None: result={"status":"MODEL_UNAVAILABLE","error":"stopped before call","score":None,"output":None}
    if result.get("status")!="success":
        result["status"]="MODEL_UNAVAILABLE"
    return {"row_key":record["row_key"],"status":result.get("status"),"result":result,
            "schema_failure":schema_failure,"attempts":client.attempts.copy(),"new_calls":client.new_calls,
            "cache_hit":bool(result.get("cache_hit",False)),"repeat_tag":repeat_tag}

def _run_batch(records,kind,*,cache_dir,cache_role,repeat_tags=None):
    outcomes={}; futures={}; first_error=None
    with ThreadPoolExecutor(max_workers=4) as pool:
        for i,record in enumerate(records):
            tag=repeat_tags[i] if repeat_tags else None
            futures[pool.submit(_run_one,kind,record,cache_dir=cache_dir,cache_role=cache_role,repeat_tag=tag)]=record["row_key"]
        for done_i,fut in enumerate(as_completed(futures),1):
            key=futures[fut]
            try: outcome=fut.result()
            except Exception as exc: outcome={"row_key":key,"status":"MODEL_UNAVAILABLE","error":f"{type(exc).__name__}: {str(exc)[:400]}","attempts":[],"new_calls":0}
            outcomes[key]=outcome
            if (kind=="jev" and outcome.get("schema_failure")) or outcome.get("status")=="STOP_CALL_BUDGET_EXCEEDED":
                ABORT.set(); first_error=outcome
                for pending in futures:
                    if pending is not fut: pending.cancel()
                break
            if done_i%25==0 or done_i==len(futures): print(f"{cache_role}: completed {done_i}/{len(futures)}")
    if first_error:
        _write_json(ART_DIR/"api_attempt_log_partial.json",ATTEMPT_LOG)
        code="STOP_FIRST_JEV_SCHEMA_FAILURE" if kind=="jev" and first_error.get("schema_failure") else (
             "STOP_FIRST_LLM_SCHEMA_FAILURE" if kind=="llm" and first_error.get("schema_failure") else "STOP_CALL_BUDGET_EXCEEDED")
        hard_stop(code,f"{cache_role} row {first_error.get('row_key')}: {first_error.get('error') or first_error.get('result',{}).get('error')}")
    return outcomes

assert_prereg_hash(); ABORT.clear()
RESCUE_RECORDS=SAMPLE.loc[SAMPLE.tier.eq("rescue")].sort_values("row_key",kind="stable").to_dict(orient="records")
jev_outcomes=_run_batch(RESCUE_RECORDS,"jev",cache_dir=JEV_CACHE,cache_role="jev")
JEV_RESULTS={k:v.get("result",{}) for k,v in jev_outcomes.items()}
jev_rows=[]
for rec in RESCUE_RECORDS:
    out=JEV_RESULTS.get(rec["row_key"],{})
    probs=out.get("probabilities") or {}
    jev_rows.append({"row_key":rec["row_key"],"fault":rec["fault"],"row_id":rec["row_id"],
       "status":out.get("status","MODEL_UNAVAILABLE"),"score":out.get("score"),
       "probability_0":probs.get("0"),"probability_1":probs.get("1"),"probability_2":probs.get("2"),
       "cost_usd_reported":out.get("cost_usd"),"cache_hit":out.get("cache_hit",False),"error":out.get("error")})
JEV_TABLE=pd.DataFrame(jev_rows).sort_values(["fault","row_key"],kind="stable")
JEV_TABLE.to_csv(ART_DIR/"jev_R2_scores.csv",index=False,encoding="utf-8")
assert_prereg_hash()
llm_outcomes=_run_batch(RESCUE_RECORDS[:20],"llm",cache_dir=LLM_CACHE,cache_role="llm_main")
FIRST20_INVALID=sum(v.get("status")!="success" for v in llm_outcomes.values())
_write_json(ART_DIR/"first20_diagnostic.json",{"rows":[r["row_key"] for r in RESCUE_RECORDS[:20]],"invalid_after_retries":FIRST20_INVALID})
_write_json(ART_DIR/"api_attempt_log_partial.json",ATTEMPT_LOG)
if FIRST20_INVALID>=3:
    hard_stop("STOP_FIRST20_LLM_SCHEMA_FAILURES",f"First 20 rows: {FIRST20_INVALID} invalid after retries; labels unopened")
llm_outcomes.update(_run_batch(RESCUE_RECORDS[20:],"llm",cache_dir=LLM_CACHE,cache_role="llm_main"))
LLM_RESULTS={k:v.get("result",{}) for k,v in llm_outcomes.items()}
MAIN_UNAVAILABLE_KEYS={k for k,v in LLM_RESULTS.items() if v.get("status")!="success"}
MAIN_UNAVAILABLE_N=len(MAIN_UNAVAILABLE_KEYS)
MAIN_UNAVAILABLE_RATE=MAIN_UNAVAILABLE_N/RESCUE_N
FORMAL_ALLOWED=MAIN_UNAVAILABLE_RATE<=.05
_write_json(ART_DIR/"availability_diagnostic.json",{"main_rescue_unavailable":MAIN_UNAVAILABLE_N,"denominator":RESCUE_N,"rate":MAIN_UNAVAILABLE_RATE,"formal_tests_allowed":FORMAL_ALLOWED})
_write_json(ART_DIR/"api_attempt_log_partial.json",ATTEMPT_LOG)
print("Main LLM unavailable:",MAIN_UNAVAILABLE_N,"/",RESCUE_N,"formal allowed:",FORMAL_ALLOWED)


## 6. Stability calls and label-free ranking freeze

Stability 5행은 07a와 동일하게 `(sequence_group,row_id)`로 정렬한 뒤 `default_rng(20260927)`로 선택했다. 각 행에 추가 LLM call 3회를 수행한다. logistic baseline은 07a train+validation dev sample/label만으로 적합한다. 그 다음 모든 고정 ranking을 저장하고 hash한 뒤에만 평가 label loader를 연다.

In [ ]:
assert_prereg_hash()
stability_records=stab_frame.loc[stab_frame.row_key.isin(STABILITY_KEYS)].to_dict(orient="records")
assert len(stability_records)==5
stability_out=[]
for rep in (1,2,3):
    ABORT.clear()
    repeat_dir=ART_DIR/"cache"/"llm_stability"/f"repeat_{rep}"
    results=_run_batch(stability_records,"llm",cache_dir=repeat_dir,cache_role="llm_stability",
                       repeat_tags=[f"repeat_{rep}"]*len(stability_records))
    for rec in stability_records:
        o=results[rec["row_key"]].get("result",{})
        stability_out.append({"row_key":rec["row_key"],"row_id":rec["row_id"],"sequence_group":rec["sequence_group"],
          "repeat_index":rep,"status":o.get("status","MODEL_UNAVAILABLE"),
          "priority_score":(o.get("output") or {}).get("priority_score"),
          "cache_hit":bool(o.get("cache_hit",False)),"cost_usd_reported":o.get("cost_usd"),"error":o.get("error")})
STABILITY=pd.DataFrame(stability_out).sort_values(["sequence_group","row_id","repeat_index"],kind="stable")
STABILITY.to_csv(ART_DIR/"llm_stability.csv",index=False,encoding="utf-8")
with (ART_DIR/"llm_noRF_results.jsonl").open("w",encoding="utf-8") as f:
    for rec in RESCUE_RECORDS:
        out=LLM_RESULTS.get(rec["row_key"],{})
        f.write(json.dumps({"row_key":rec["row_key"],"row_id":rec["row_id"],"fault":rec["fault"],
             "status":out.get("status","MODEL_UNAVAILABLE"),"output":out.get("output"),
             "citation_count":out.get("citation_count"),"citation_agreement":out.get("citation_agreement"),
             "cost_usd_reported":out.get("cost_usd"),"cache_hit":out.get("cache_hit",False),"error":out.get("error")},
             ensure_ascii=False,allow_nan=False)+"\n")
with (ART_DIR/"api_attempt_log.jsonl").open("w",encoding="utf-8") as f:
    for rec in ATTEMPT_LOG: f.write(json.dumps(rec,ensure_ascii=False,allow_nan=False)+"\n")

# 07a train+validation only; eval test sequence groups are asserted disjoint.
NO_RF_FIELDS=[f for f in PACKET_FIELDS if f!="rf_probability"]
def _feature_vector(packet):
    f=packet["fields"]
    return [float(f[name]) for name in NO_RF_FIELDS]
DEV_SAMPLE=pd.read_csv(P07A_DIR/"dev_sample.csv",usecols=["row_id","sequence_group","packet_status","packet_json"])
DEV_LABELS=pd.read_csv(P07A_DIR/"dev_labels.csv",usecols=["row_id","sequence_group","label_near_5k"])
if set(DEV_SAMPLE.sequence_group)&set(SAMPLE.sequence_group):
    hard_stop("STOP_LABEL_OR_IDENTIFIER_LEAK","A test sequence group occurs in 07a train/validation dev data.")
DEV=DEV_SAMPLE.loc[DEV_SAMPLE.packet_status.eq("READY")].merge(
    DEV_LABELS,on=["row_id","sequence_group"],validate="one_to_one")
if DEV.empty or DEV.label_near_5k.nunique()!=2:
    hard_stop("BLOCKED_SPEC_AMBIGUITY","07a dev READY data cannot fit the preregistered logistic baseline.")
DEV_X=np.asarray([_feature_vector(json.loads(s)) for s in DEV.packet_json],dtype=float)
DEV_Y=DEV.label_near_5k.to_numpy(dtype=int)
LOGISTIC=make_pipeline(StandardScaler(),LogisticRegression(C=1,solver="lbfgs",max_iter=5000))
LOGISTIC.fit(DEV_X,DEV_Y)

RESCUE=SAMPLE.loc[SAMPLE.tier.eq("rescue")].copy().sort_values(["fault","row_key"],kind="stable").reset_index(drop=True)
RESCUE["jev_score"]=[JEV_RESULTS.get(k,{}).get("score") for k in RESCUE.row_key]
RESCUE["jev_status"]=[JEV_RESULTS.get(k,{}).get("status","MODEL_UNAVAILABLE") for k in RESCUE.row_key]
RESCUE["llm_status"]=[LLM_RESULTS.get(k,{}).get("status","MODEL_UNAVAILABLE") for k in RESCUE.row_key]
RESCUE["llm_priority"]=[(LLM_RESULTS.get(k,{}).get("output") or {}).get("priority_score") for k in RESCUE.row_key]
RESCUE["lstm_priority"]=-RESCUE.lstm_pred_seconds.astype(float)
RESCUE["logistic_probability"]=LOGISTIC.predict_proba(np.asarray([
 _feature_vector(json.loads(s)) for s in RESCUE.packet_json],dtype=float))[:,1]

def round_half_up_count(n,fraction): return int(math.floor(int(n)*float(fraction)+0.5))
def _sort_keys(df,score,descending=True,random_tie=None):
    work=df.copy(); work["_score"] = pd.to_numeric(work[score],errors="coerce")
    work["_missing"]=work["_score"].isna()
    if random_tie is not None: work["_random_tie"]=random_tie
    else: work["_random_tie"]=0.0
    work=work.sort_values(["_missing","_score","_random_tie","row_key"],
        ascending=[True,not descending,True,True],kind="stable",na_position="last")
    return work.row_key.tolist()

def _rank_split(df,score,*,llm_in_mid=False,random_ties=None):
    valid=df.loc[pd.to_numeric(df[score],errors="coerce").notna()].copy()
    invalid=df.loc[pd.to_numeric(df[score],errors="coerce").isna()].copy()
    ordered=_sort_keys(valid,score,True if score=="lstm_priority" else True)
    # Scores are stored as direct priority values, larger is always higher.
    ordered=_sort_keys(valid,score,descending=True)
    h=round_half_up_count(len(ordered),.25); m=round_half_up_count(len(ordered),.50)
    if h+m>len(ordered): m=max(0,len(ordered)-h)
    high=ordered[:h]; mid=ordered[h:h+m]; low=ordered[h+m:]
    score_map=df.set_index("row_key")[score].to_dict()
    llm_map=df.set_index("row_key").llm_priority.to_dict()
    if llm_in_mid:
        mid_df=df.loc[df.row_key.isin(mid)].copy()
        if random_ties is not None: tie={k:v for k,v in zip(df.row_key,random_ties)}; mid_df["_tie"]=mid_df.row_key.map(tie)
        else: mid_df["_tie"]=0.0
        mid_df["_llm_missing"]=pd.to_numeric(mid_df.llm_priority,errors="coerce").isna()
        mid_df["_llm"]=pd.to_numeric(mid_df.llm_priority,errors="coerce")
        mid=mid_df.sort_values(["_llm_missing","_llm","_tie",score,"row_key"],
             ascending=[True,False,True,False,True],kind="stable",na_position="last").row_key.tolist()
    high_order=_sort_keys(df.loc[df.row_key.isin(high)],score,descending=True)
    low_valid=[k for k in low if pd.notna(score_map.get(k))]
    low_order=_sort_keys(df.loc[df.row_key.isin(low_valid)],score,descending=True)+sorted(set(low)-set(low_valid))
    # JEV unavailable scores are LOW, after all valid LOW rows.
    low_order += sorted(invalid.row_key.tolist())
    return high+mid+low_order,{k:"HIGH" for k in high}|{k:"MID" for k in mid}|{k:"LOW" for k in low_order}

rankings={}; route_tiers={}; rank_scores={}
for fault in FAULTS:
    sub=RESCUE.loc[RESCUE.fault.eq(fault)].copy()
    jev_order,jev_tier=_rank_split(sub,"jev_score",llm_in_mid=True)
    rankings.setdefault("JEV_routing",[]).extend(jev_order); route_tiers.setdefault("JEV_routing",{}).update(jev_tier)
    full_order=_sort_keys(sub,"llm_priority",descending=True)
    rankings.setdefault("full_LLM",[]).extend(full_order)
    lstm_order,lstm_tier=_rank_split(sub,"lstm_priority",llm_in_mid=True)
    rankings.setdefault("LSTM_routing",[]).extend(lstm_order); route_tiers.setdefault("LSTM_routing",{}).update(lstm_tier)
    rankings.setdefault("JEV_only",[]).extend(_sort_keys(sub,"jev_score",descending=True))
    rankings.setdefault("LSTM_only",[]).extend(_sort_keys(sub,"lstm_priority",descending=True))
    rankings.setdefault("logistic",[]).extend(_sort_keys(sub,"logistic_probability",descending=True))
    route_tiers.setdefault("LSTM_routing",{}).update(lstm_tier)
    for method,col in [("JEV_routing","jev_score"),("full_LLM","llm_priority"),
                       ("LSTM_routing","lstm_priority"),("JEV_only","jev_score"),
                       ("LSTM_only","lstm_priority"),("logistic","logistic_probability")]:
        rank_scores.setdefault(method,{}).update(sub.set_index("row_key")[col].to_dict())

# Budget-matched 1,000 random routes, with deterministic seeded assignment and tie order.
RNG=np.random.default_rng(RANDOM_SEED+1); RANDOM_RANKINGS=[]; random_rank_rows=[]; random_mid_counts={}
SEQGROUP_BY_KEY=RESCUE.set_index("row_key").sequence_group.to_dict()
for fault in FAULTS:
    sub=RESCUE.loc[RESCUE.fault.eq(fault)].sort_values("row_key",kind="stable").reset_index(drop=True)
    jev_mid_count=sum(route_tiers["JEV_routing"].get(k)=="MID" for k in sub.row_key)
    random_mid_counts[fault]=int(jev_mid_count)
    for rep in range(1000):
        idx=np.arange(len(sub)); mid_idx=np.sort(RNG.choice(idx,size=jev_mid_count,replace=False)) if jev_mid_count else np.array([],dtype=int)
        mid_set=set(sub.iloc[mid_idx].row_key.tolist()); remaining=[k for k in sub.row_key if k not in mid_set]
        RNG.shuffle(remaining); nh=round_half_up_count(len(remaining),.50)
        high=remaining[:nh]; low=remaining[nh:]
        tievals=RNG.random(len(sub)); tie_map=dict(zip(sub.row_key,tievals))
        mids=sub.loc[sub.row_key.isin(mid_set)].copy(); mids["_tie"]=mids.row_key.map(tie_map)
        mids["_missing"]=pd.to_numeric(mids.llm_priority,errors="coerce").isna(); mids["_score"]=pd.to_numeric(mids.llm_priority,errors="coerce")
        mids=mids.sort_values(["_missing","_score","_tie","row_key"],ascending=[True,False,True,True],kind="stable",na_position="last")
        order=list(high)+mids.row_key.tolist()+list(low)
        ranks={k:i+1 for i,k in enumerate(order)}; RANDOM_RANKINGS.append({"fault":fault,"rep":rep,"ranks":ranks})
        random_rank_rows.extend({"row_key":k,"fault":fault,"sequence_group":str(SEQGROUP_BY_KEY[k]),
          "method":f"random_{rep+1:04d}","rank":ranks[k],"queue_tier":"HIGH" if k in high else ("MID" if k in mid_set else "LOW"),
          "score":None,"status":"randomized"} for k in order)

# Compose and freeze a long-form table containing all 1,000 random rank realizations.
RANK_ROWS=[]
for method,order in rankings.items():
    tier_map=route_tiers.get(method,{})
    for rank,key in enumerate(order,1):
        row=RESCUE.loc[RESCUE.row_key.eq(key)].iloc[0]
        RANK_ROWS.append({"row_key":key,"fault":row.fault,"sequence_group":row.sequence_group,
          "method":method,"rank":rank,"queue_tier":tier_map.get(key,""),
          "score":rank_scores.get(method,{}).get(key),
          "status":row.jev_status if method.startswith("JEV") else (row.llm_status if method in {"full_LLM","LSTM_routing"} else "success")})
RANK_ROWS.extend(random_rank_rows)
# RANKINGS_FINALIZATION
RANKING_BY_FAULT={}
for method,all_order in rankings.items():
    RANKING_BY_FAULT[method]={}
    for fault in FAULTS:
        RANKING_BY_FAULT[method][fault]=[k for k in all_order if RESCUE.loc[RESCUE.row_key.eq(k),"fault"].iloc[0]==fault]
RANK_ROWS=[]
for method,by_fault in RANKING_BY_FAULT.items():
    tier_map=route_tiers.get(method,{})
    for fault,order in by_fault.items():
        for rank,key in enumerate(order,1):
            row=RESCUE.loc[RESCUE.row_key.eq(key)].iloc[0]
            RANK_ROWS.append({"row_key":key,"fault":fault,"sequence_group":row.sequence_group,
              "method":method,"rank":rank,"queue_tier":tier_map.get(key,""),
              "score":rank_scores.get(method,{}).get(key),
              "status":row.jev_status if method.startswith("JEV") else (row.llm_status if method in {"full_LLM","LSTM_routing"} else "success")})
RANK_ROWS.extend(random_rank_rows)
RANKING_TABLE=pd.DataFrame(RANK_ROWS).sort_values(["method","fault","rank","row_key"],kind="stable")
RANKINGS_FROZEN=False; LABELS_LOADED=False
RANKING_PATH=ART_DIR/"rankings.csv"
RANKING_TABLE.to_csv(RANKING_PATH,index=False,encoding="utf-8")
RANKINGS_SHA256=file_sha256(RANKING_PATH)
RANKINGS_FROZEN=True
_append_log(f"\n- rankings frozen UTC: {datetime.now(timezone.utc).isoformat()}\n- rankings SHA-256: {RANKINGS_SHA256}\n")
assert_prereg_hash()
print("rankings frozen: rows",len(RANKING_TABLE),"SHA-256",RANKINGS_SHA256)
print("JEV MID budgets by fault:",random_mid_counts,"; labels remain unloaded:",not LABELS_LOADED)

## 7. Ranking freeze 이후 label 공개 및 confirmatory feasibility

`rankings.csv`가 저장되고 SHA-256이 고정된 뒤에만 test `wall_ttf_seconds`를 읽는다. 먼저 exact 06a 80행 confusion을 재계산한다. 이후 eval label 별도 파일과 positive prevalence를 기록한다.

In [ ]:
# LABELS_LOAD
LABELS_FIRST_LOAD_UTC=datetime.now(timezone.utc).isoformat()

def load_eval_labels():
    global LABELS_LOADED
    if not RANKINGS_FROZEN: raise RuntimeError("STOP_LABEL_OR_IDENTIFIER_LEAK: rankings are not frozen")
    if file_sha256(RANKING_PATH)!=RANKINGS_SHA256: hard_stop("STOP_PREREG_HASH_MISMATCH","rankings.csv bytes changed before label load.")
    if LABELS_LOADED: raise RuntimeError("Evaluation labels may be loaded only once.")
    parts=[]
    for fault in FAULTS:
        label_source=pd.read_parquet(PRED_DIR/f"{fault}_test_predictions.parquet",
             columns=["sequence_id","raw_row_index","wall_ttf_seconds"])
        label_source["sequence_id"]=label_source.sequence_id.astype(str)
        if not np.isfinite(label_source.wall_ttf_seconds.to_numpy(float)).all():
            hard_stop("BLOCKED_SPEC_AMBIGUITY",f"{fault}: nonfinite test TTF labels.")
        req=SAMPLE.loc[SAMPLE.fault.eq(fault),["row_key","sequence_id","original_row_position"]].copy()
        matched=req.merge(label_source,left_on=["sequence_id","original_row_position"],
             right_on=["sequence_id","raw_row_index"],how="left",validate="one_to_one")
        if len(matched)!=len(req) or matched.wall_ttf_seconds.isna().any():
            hard_stop("BLOCKED_SPEC_AMBIGUITY",f"{fault}: sampled evaluation labels could not be joined one-to-one.")
        matched["fault"]=fault
        matched["label_near_5k"]=(matched.wall_ttf_seconds.astype(float)<=ALARM_LIMIT_SECONDS).astype(int)
        parts.append(matched[["row_key","fault","sequence_id","original_row_position","wall_ttf_seconds","label_near_5k"]])
    labels=pd.concat(parts,ignore_index=True)
    if labels.row_key.duplicated().any() or len(labels)!=len(SAMPLE):
        hard_stop("STOP_LABEL_OR_IDENTIFIER_LEAK","Evaluation label identities do not exactly match frozen sample.")
    # Reproduce the preserved 06a result from its exact stored row identities.
    fault3_labels=pd.read_parquet(PRED_DIR/"fault3_test_predictions.parquet",
             columns=["sequence_id","raw_row_index","wall_ttf_seconds"])
    fault3_labels["sequence_id"]=fault3_labels.sequence_id.astype(str)
    old=REPRO80_KEYS.merge(pilot_safe,on=keys,how="left",validate="one_to_one").merge(
       fault3_labels,left_on=keys,right_on=["sequence_id","raw_row_index"],how="left",validate="one_to_one")
    if old.wall_ttf_seconds.isna().any(): hard_stop("STOP_04B_REPRODUCTION_FAILED","06a labels missing for a stored test identity.")
    y=old.wall_ttf_seconds.to_numpy(float)<=ALARM_LIMIT_SECONDS
    pred=old.pipeline_alarm.astype(str).str.lower().eq("true").to_numpy()
    confusion={"TP":int(np.sum(pred&y)),"FP":int(np.sum(pred&~y)),
               "FN":int(np.sum(~pred&y)),"TN":int(np.sum(~pred&~y))}
    if any(confusion[k]!=REPRO80_EXPECTED[k] for k in ("TP","FP","FN")):
        hard_stop("STOP_04B_REPRODUCTION_FAILED",f"06a exact 80-row regression mismatch: {confusion}")
    labels.to_csv(ART_DIR/"eval_labels.csv",index=False,encoding="utf-8")
    LABELS_LOADED=True
    return labels,confusion

assert_prereg_hash()
EVAL_LABELS,REPRO80_ACTUAL=load_eval_labels()
EVAL=SAMPLE.merge(EVAL_LABELS[["row_key","label_near_5k"]],on="row_key",validate="one_to_one")
print("06a exact 80-row regression PASS:",REPRO80_ACTUAL)
print("evaluation labels first loaded UTC:",datetime.now(timezone.utc).isoformat())
prevalence_table=EVAL.groupby("fault",sort=True).agg(rows=("row_key","size"),positive_rows=("label_near_5k","sum"),
 positive_rate=("label_near_5k","mean"),sequences=("sequence_id","nunique")).reset_index()
display(prevalence_table)
primary=EVAL.loc[EVAL.fault.isin(["fault1","fault2"])&EVAL.tier.eq("rescue")]
primary_pos=int(primary.label_near_5k.sum()); primary_neg=int(len(primary)-primary_pos)
primary_seq=int(primary.sequence_id.nunique())
FEASIBLE=(primary_pos>=30 and primary_neg>=30 and primary_seq>=10)
FEASIBILITY_STATUS="PASS" if FEASIBLE else "INSUFFICIENT_CONFIRMATORY_SAMPLE"
print("fault1+fault2 rescue feasibility:",{"positive_rows":primary_pos,"negative_rows":primary_neg,
 "rescue_sequences":primary_seq,"status":FEASIBILITY_STATUS})

## 8. 고정 분석, gatekeeping, effect size, 비용

Precision은 fault별 rescue의 상위 K행에서 계산하고, pooled 값은 fault별 E budget에서 발견한 positives와 reviewed rows를 합산한다. 비용은 `총 비용 = 검토 행 수 × 1 + 미발견 imminent failure 수 × r`로 계산한다. effect size는 기존 04b tier1 positives에 rescue top-E 회수를 더해 전체 sampled positives 중 recall을 구한다.

MODEL_UNAVAILABLE 행을 주 분석에 유지합니다. 별도 민감도 분석은 그 행을 모든 고정 순위에서 제외하고 상대 순서를 유지해 E20과 H1~H3를 다시 계산합니다. 판정은 주 분석에만 근거하며 순차 검정 규칙도 유지합니다.

In [ ]:
assert_prereg_hash()
from scipy.stats import hypergeom
from collections import Counter
Y_MAP=dict(zip(EVAL_LABELS.row_key,EVAL_LABELS.label_near_5k.astype(int)))
GROUP_MAP=dict(zip(RESCUE.row_key,RESCUE.sequence_group))
FAULT_MAP=dict(zip(RESCUE.row_key,RESCUE.fault))
JEV_TIER=route_tiers["JEV_routing"]
RANDOM_BY_FAULT_REP={(x["fault"],x["rep"]):x["ranks"] for x in RANDOM_RANKINGS}

def _order(method,fault,rep=None):
    if method=="random":
        rd=RANDOM_BY_FAULT_REP[(fault,int(rep))]
        return sorted(rd,key=lambda k:(rd[k],k))
    return RANKING_BY_FAULT[method][fault]

def _top(method,fault,fraction,rep=None):
    order=_order(method,fault,rep)
    k=round_half_up_count(len(order),fraction)
    return order[:k]

def _precision(method,faults,fraction,rep=None):
    positive=reviewed=0
    for fault in faults:
        selected=_top(method,fault,fraction,rep)
        reviewed+=len(selected); positive+=sum(Y_MAP[k] for k in selected)
    return (positive/reviewed if reviewed else float("nan")),positive,reviewed

PRIMARY_FAULTS=("fault1","fault2")
H1_EST,H1_POS,H1_K=_precision("JEV_routing",PRIMARY_FAULTS,.20)
H2_LLM,H2_POS,H2_K=_precision("full_LLM",PRIMARY_FAULTS,.20)
PRIMARY_BASE=primary_pos/len(primary)
random_precision_by_rep=[]
for rep in range(1000):
    p,pos,k=_precision("random",PRIMARY_FAULTS,.20,rep)
    random_precision_by_rep.append(p)
H3_RANDOM=float(np.mean(random_precision_by_rep))

# Sequence-cluster bootstrap. Duplicate selected sequences with all sampled rows;
# the ranking order itself remains frozen and label-blind.
BOOT_REPS=10000
BOOT_DISTRIBUTIONS={"H1":[],"H2":[],"H3":[]}
if FEASIBLE and FORMAL_ALLOWED:
    group_keys={f:sorted(RESCUE.loc[RESCUE.fault.eq(f),"sequence_group"].unique().tolist()) for f in PRIMARY_FAULTS}
    rows_by_fault={f:RESCUE.loc[RESCUE.fault.eq(f),"row_key"].tolist() for f in PRIMARY_FAULTS}
    j_order={f:RANKING_BY_FAULT["JEV_routing"][f] for f in PRIMARY_FAULTS}
    l_order={f:RANKING_BY_FAULT["full_LLM"][f] for f in PRIMARY_FAULTS}
    boot_rng=np.random.default_rng(RANDOM_SEED)
    for boot_i in range(BOOT_REPS):
        pooled_j_pos=pooled_j_k=pooled_l_pos=pooled_l_k=0
        pooled_y=pooled_n=pooled_rand_exp=pooled_rand_k=0.0
        for fault in PRIMARY_FAULTS:
            groups=group_keys[fault]
            sampled_groups=boot_rng.choice(np.asarray(groups,dtype=object),size=len(groups),replace=True).tolist()
            mult=Counter(sampled_groups)
            def expand(order):
                return [key for key in order for _ in range(mult.get(GROUP_MAP[key],0))]
            ej=expand(j_order[fault]); el=expand(l_order[fault]); all_rows=expand(rows_by_fault[fault])
            n=len(all_rows); k=round_half_up_count(n,.20)
            yp=sum(Y_MAP[x] for x in all_rows); prevalence=yp/n if n else 0.0
            pooled_y+=yp; pooled_n+=n
            pooled_j_pos+=sum(Y_MAP[x] for x in ej[:k]); pooled_j_k+=min(k,len(ej))
            pooled_l_pos+=sum(Y_MAP[x] for x in el[:k]); pooled_l_k+=min(k,len(el))
            b=sum(mult.get(GROUP_MAP[key],0) for key in rows_by_fault[fault] if JEV_TIER.get(key)=="MID")
            h=round_half_up_count(n-b,.50)
            if k<=h:
                exp_random=k*prevalence
            else:
                q_mid=min(max(0,k-h),b)
                exp_random=h*prevalence
                if q_mid and b:
                    # Uniform random MID subset; within it, the frozen LLM ordering is used.
                    positions=np.arange(n,dtype=int)
                    probs=(b/n)*hypergeom.cdf(q_mid-1,n-1,positions,b-1)
                    y_llm=np.asarray([Y_MAP[x] for x in el],dtype=float)
                    exp_random+=float(np.dot(probs,y_llm))
                low_take=max(0,k-h-b)
                if low_take: exp_random+=low_take*prevalence
            pooled_rand_exp+=exp_random; pooled_rand_k+=k
        jp=pooled_j_pos/pooled_j_k if pooled_j_k else 0.0
        lp=pooled_l_pos/pooled_l_k if pooled_l_k else 0.0
        base=pooled_y/pooled_n if pooled_n else 0.0
        rp=pooled_rand_exp/pooled_rand_k if pooled_rand_k else 0.0
        BOOT_DISTRIBUTIONS["H1"].append(jp-base)
        BOOT_DISTRIBUTIONS["H2"].append(jp-lp)
        BOOT_DISTRIBUTIONS["H3"].append(jp-rp)

def _lower(values): return float(np.quantile(np.asarray(values,dtype=float),.05)) if values else None
H1_LOWER=_lower(BOOT_DISTRIBUTIONS["H1"]); H2_LOWER=None; H3_LOWER=None
if not FORMAL_ALLOWED:
    H1_DECISION=H2_DECISION=H3_DECISION="DESCRIPTIVE_ONLY_MODEL_UNAVAILABLE"
    H1_GATE=H2_GATE=H3_GATE="More than 5% main rescue LLM unavailable; no formal testing"
elif not FEASIBLE:
    H1_DECISION=H2_DECISION=H3_DECISION="INSUFFICIENT_CONFIRMATORY_SAMPLE"
    H1_GATE=H2_GATE=H3_GATE="sample feasibility condition not met; formal test not run"
else:
    H1_DECISION="PASS" if H1_LOWER>0 else "FAIL_TO_REJECT"
    H1_GATE="TESTED"
    if H1_DECISION!="PASS":
        H2_DECISION=H3_DECISION="NOT_TESTED_DUE_TO_GATEKEEPING"
        H2_GATE=H3_GATE="H1 did not pass"
    else:
        H2_LOWER=_lower(BOOT_DISTRIBUTIONS["H2"])
        H2_DECISION="PASS" if H2_LOWER>-.05 else "FAIL_TO_REJECT"
        H2_GATE="TESTED"
        if H2_DECISION!="PASS":
            H3_DECISION="NOT_TESTED_DUE_TO_GATEKEEPING"; H3_GATE="H2 did not pass"
        else:
            H3_LOWER=_lower(BOOT_DISTRIBUTIONS["H3"])
            H3_DECISION="PASS" if H3_LOWER>0 else "FAIL_TO_REJECT"; H3_GATE="TESTED"
HYPOTHESIS_TESTS={
 "primary_population":"fault1+fault2 combined rescue","primary_budget":"E20 per fault, half-up",
 "alpha":.05,"bootstrap_repetitions":BOOT_REPS,"bootstrap_seed":RANDOM_SEED,"random_routing_seed":RANDOM_SEED+1,
 "bootstrap_unit":"sequence within fault; all rescue rows retained per sampled sequence",
 "feasibility":{"positive_rows":primary_pos,"negative_rows":primary_neg,"rescue_sequences":primary_seq,
                 "status":FEASIBILITY_STATUS},
 "H1":{"hypothesis":"JEV precision@E20 - rescue prevalence > 0","population":"fault1+fault2 rescue",
  "budget":"E20","estimate":float(H1_EST),"comparator_estimate":float(PRIMARY_BASE),
  "difference":float(H1_EST-PRIMARY_BASE),"bootstrap_repetitions":BOOT_REPS if FEASIBLE and FORMAL_ALLOWED else 0,
  "alpha":.05,"one_sided_lower_bound":H1_LOWER,"threshold":0.0,"decision":H1_DECISION,
  "testing_order":1,"gatekeeping_status":H1_GATE},
 "H2":{"hypothesis":"JEV precision@E20 - full LLM precision@E20 > -0.05","population":"fault1+fault2 rescue",
  "budget":"E20","estimate":float(H1_EST),"comparator_estimate":float(H2_LLM),
  "difference":float(H1_EST-H2_LLM),"bootstrap_repetitions":BOOT_REPS if FEASIBLE and H2_GATE=="TESTED" else 0,
  "alpha":.05,"one_sided_lower_bound":H2_LOWER,"threshold":-.05,"noninferiority_margin":-.05,
  "decision":H2_DECISION,"testing_order":2,"gatekeeping_status":H2_GATE},
 "H3":{"hypothesis":"JEV precision@E20 - mean random routing precision@E20 > 0","population":"fault1+fault2 rescue",
  "budget":"E20","estimate":float(H1_EST),"comparator_estimate":float(H3_RANDOM),
  "difference":float(H1_EST-H3_RANDOM),"bootstrap_repetitions":BOOT_REPS if FEASIBLE and H3_GATE=="TESTED" else 0,
  "alpha":.05,"one_sided_lower_bound":H3_LOWER,"threshold":0.0,"decision":H3_DECISION,
  "testing_order":3,"gatekeeping_status":H3_GATE,
  "random_routing_repetitions":1000,"bootstrap_random_comparator":"exact conditional expectation of preregistered randomized HIGH/MID/LOW route; point estimate is mean of 1,000 seeded routes"}
}
_write_json(ART_DIR/"hypothesis_tests.json",HYPOTHESIS_TESTS)

# 04b effect size: rescue recovery and recall over every sampled LSTM-alarm endpoint.
SCOPE_FAULTS={"fault1":["fault1"],"fault2":["fault2"],"fault3":["fault3"],"fault1+fault2":["fault1","fault2"]}
def _effect_row(scope,faults,fraction):
    eval_scope=EVAL.loc[EVAL.fault.isin(faults)]
    rescue_scope=eval_scope.loc[eval_scope.tier.eq("rescue")]
    tier1_scope=eval_scope.loc[eval_scope.tier.eq("tier1")]
    selected=[k for fault in faults for k in _top("JEV_routing",fault,fraction)]
    found=sum(Y_MAP[k] for k in selected); rescue_pos=int(rescue_scope.label_near_5k.sum())
    all_pos=int(eval_scope.label_near_5k.sum()); tier1_pos=int(tier1_scope.label_near_5k.sum())
    return {"scope":scope,"E":int(round(fraction*100)),"additional_review_rows":len(selected),
      "rescue_positive_rows":rescue_pos,"imminent_failures_recovered":int(found),
      "rescue_recovery_rate":float(found/rescue_pos) if rescue_pos else None,
      "rescue_precision":float(found/len(selected)) if selected else None,
      "tier1_positive_rows":tier1_pos,"total_sampled_positive_rows":all_pos,
      "04b_recall":float(tier1_pos/all_pos) if all_pos else None,
      "structural_recall":float((tier1_pos+found)/all_pos) if all_pos else None,
      "recall_change_vs_04b":float(found/all_pos) if all_pos else None}
EFFECT_ROWS=[_effect_row(scope,faults,f) for scope,faults in SCOPE_FAULTS.items() for f in (.10,.20,.30)]
EFFECT=pd.DataFrame(EFFECT_ROWS).sort_values(["scope","E"],kind="stable")
EFFECT.to_csv(ART_DIR/"effect_vs_04b.csv",index=False,encoding="utf-8")

# Cost curve for 04b only and 04b plus the JEV structural rescue lane.
COST_ROWS=[]
for scope,faults in SCOPE_FAULTS.items():
    es=EVAL.loc[EVAL.fault.isin(faults)]; rescue_es=es.loc[es.tier.eq("rescue")]; tier_es=es.loc[es.tier.eq("tier1")]
    total_pos=int(es.label_near_5k.sum()); tier_pos=int(tier_es.label_near_5k.sum())
    strategies=[("04b_only",None,0)]
    for frac in (.10,.20,.30):
        selected=[k for fault in faults for k in _top("JEV_routing",fault,frac)]
        strategies.append((f"04b_plus_structural_E{int(frac*100)}",frac,sum(Y_MAP[k] for k in selected)))
    base_reviews=len(tier_es); base_missed=total_pos-tier_pos
    for name,frac,lane_found in strategies:
        lane_rows=0 if frac is None else sum(len(_top("JEV_routing",f,frac)) for f in faults)
        reviews=base_reviews+lane_rows; missed=total_pos-tier_pos-lane_found
        for r in range(1,21):
            breakeven=None if frac is None or lane_found==0 else float(lane_rows/lane_found)
            COST_ROWS.append({"scope":scope,"strategy":name,"missed_failure_cost_r":r,"review_count":reviews,
              "missed_positive_count":missed,"review_cost":float(reviews),"missed_failure_cost":float(missed*r),
              "total_cost":float(reviews+missed*r),"incremental_reviews_vs_04b":lane_rows,
              "incremental_recovered_positives_vs_04b":int(lane_found),"break_even_r_vs_04b":breakeven})
COST_CURVE=pd.DataFrame(COST_ROWS)
COST_CURVE.to_csv(ART_DIR/"cost_curve.csv",index=False,encoding="utf-8")

# Exploratory comparisons only; no extra p-values or hypotheses.
EXPL_ROWS=[]
def _add_expl(scope,faults,method,comparator,fraction):
    p1,pos1,k1=_precision(method,faults,fraction); p2,pos2,k2=_precision(comparator,faults,fraction)
    EXPL_ROWS.append({"analysis_type":"탐색","scope":scope,"E":int(fraction*100),"method":method,
      "comparator":comparator,"precision":p1,"comparator_precision":p2,"difference":p1-p2,
      "positive_rows":pos1,"reviewed_rows":k1,"formal_test":False})
for scope,faults in SCOPE_FAULTS.items():
    for frac in (.10,.20,.30):
        for method,comp in [("JEV_routing","LSTM_routing"),("full_LLM","logistic"),
                            ("full_LLM","LSTM_only"),("JEV_only","LSTM_only")]:
            _add_expl(scope,faults,method,comp,frac)
# Stability summaries include main plus the three independent repeat calls.
for key,group in STABILITY.groupby("row_key",sort=True):
    main=(LLM_RESULTS.get(key,{}).get("output") or {}).get("priority_score")
    values=[float(x) for x in group.priority_score.dropna().tolist()]
    if main is not None: values=[float(main)]+values
    EXPL_ROWS.append({"analysis_type":"탐색","scope":"stability","E":None,"method":"LLM_priority_score",
       "comparator":"main_vs_three_repeats","difference":float(np.std(values,ddof=1)) if len(values)>1 else None,
       "precision":float(main) if main is not None else None,"comparator_precision":float(np.mean(values)) if values else None,
       "positive_rows":None,"reviewed_rows":len(values),"formal_test":False,"row_key":key,
       "score_range":float(max(values)-min(values)) if values else None})
EXPLORATORY=pd.DataFrame(EXPL_ROWS)
EXPLORATORY.to_csv(ART_DIR/"exploratory.csv",index=False,encoding="utf-8")

# API call/cost accounting: provider-reported usage when available, 07a mean-cost fallback otherwise.
attempt_df=pd.DataFrame(ATTEMPT_LOG)
if not attempt_df.empty:
    api_attempts=attempt_df.loc[~attempt_df.cache_hit.astype(bool)].copy()
    retry_groups=api_attempts.groupby(["row_key","call_stage","repeat_tag"],dropna=False).size()
    retry_count=int((retry_groups-1).clip(lower=0).sum())
    unavailable_count=int(sum((JEV_TABLE.status!="success").astype(int))+sum(LLM_RESULTS[k].get("status","MODEL_UNAVAILABLE")!="success" for k in LLM_RESULTS)+
        sum((STABILITY.status!="success").astype(int)))
else:
    api_attempts=pd.DataFrame(); retry_count=0; unavailable_count=0
api_actual={"jev":0.0,"llm_main":0.0,"llm_stability":0.0}; api_fallback={"jev":0.0,"llm_main":0.0,"llm_stability":0.0}
for _,a in api_attempts.iterrows():
    role=str(a.call_stage); unit=jev_unit if role=="jev" else float(llm_unit)
    value=a.get("cost_usd")
    if isinstance(value,(int,float,np.number)) and pd.notna(value): api_actual[role]=api_actual.get(role,0.0)+float(value)
    else: api_fallback[role]=api_fallback.get(role,0.0)+float(unit)

LLM_EQ_COST={}
for key,out in LLM_RESULTS.items():
    value=out.get("cost_usd")
    LLM_EQ_COST[key]=float(value) if isinstance(value,(int,float)) and math.isfinite(float(value)) else float(llm_unit)
JEV_EQ_COST={}
for key,out in JEV_RESULTS.items():
    value=out.get("cost_usd")
    JEV_EQ_COST[key]=float(value) if isinstance(value,(int,float)) and math.isfinite(float(value)) else float(jev_unit)
mid_keys=[k for fault in FAULTS for k in RANKING_BY_FAULT["JEV_routing"][fault]
          if route_tiers["JEV_routing"].get(k)=="MID"]
full_llm_equiv_cost=float(sum(LLM_EQ_COST.values()))
routed_llm_equiv_cost=float(sum(LLM_EQ_COST[k] for k in mid_keys))
jev_equiv_cost=float(sum(JEV_EQ_COST.values()))
routed_total_equiv_cost=jev_equiv_cost+routed_llm_equiv_cost
llm_reduction_pct=float(100*(1-len(mid_keys)/len(RESCUE))) if len(RESCUE) else 0.0
saving_usd=full_llm_equiv_cost-routed_total_equiv_cost
saving_pct=float(100*saving_usd/full_llm_equiv_cost) if full_llm_equiv_cost else None
budget_snapshot=COST_BUDGET.snapshot()
CALL_SUMMARY={
 "expected_logical_calls":{"JEV":EXPECTED_JEV,"main_LLM":EXPECTED_MAIN_LLM,"stability_LLM":EXPECTED_STABILITY,"total":EXPECTED_TOTAL},
 "actual_API_requests":{"JEV":int(CALL_COUNTS["jev"]),"main_LLM":int(CALL_COUNTS["llm_main"]),
                         "stability_LLM":int(CALL_COUNTS["llm_stability"]),"total":int(sum(CALL_COUNTS.values()))},
 "retry_count":retry_count,"unavailable_outputs":unavailable_count,
 "projected_full_LLM_equivalent_calls":int(len(RESCUE)),"JEV_routing_LLM_equivalent_calls":int(len(mid_keys),),
 "LLM_call_reduction_pct":llm_reduction_pct,
 "07b_API_cost_reported_usd":float(sum(api_actual.values())),
 "07b_API_cost_fallback_estimated_usd":float(sum(api_fallback.values())),
 "07b_API_cost_by_stage_reported_usd":api_actual,"07b_API_cost_by_stage_fallback_usd":api_fallback,
 "equivalent_full_LLM_response_cost_usd":full_llm_equiv_cost,
 "equivalent_routed_LLM_cost_usd":routed_llm_equiv_cost,"equivalent_JEV_R2_cost_usd":jev_equiv_cost,
 "equivalent_JEV_plus_routed_LLM_cost_usd":routed_total_equiv_cost,
 "estimated_monetary_saving_usd":float(saving_usd),"estimated_saving_pct":saving_pct,
 "pre_call_estimated_cost_usd":{"JEV":EST_JEV,"LLM":EST_LLM,"total":EST_TOTAL,"cap":COST_CAP},
 "actual_cost_guard":budget_snapshot}

_write_json(ART_DIR/"hypothesis_tests.json",HYPOTHESIS_TESTS)

def _plain(value):
    if isinstance(value,dict): return {str(k):_plain(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)): return [_plain(v) for v in value]
    if isinstance(value,np.generic): return _plain(value.item())
    if value is pd.NA or (isinstance(value,float) and not math.isfinite(value)): return None
    if isinstance(value,(str,int,float,bool)) or value is None: return value
    return str(value)

def _rows_plain(frame): return [_plain(r) for r in frame.to_dict(orient="records")]
per_fault_summary={}
for fault in FAULTS:
    sf=SAMPLE.loc[SAMPLE.fault.eq(fault)]
    per_fault_summary[fault]={"test_sequences":int(len(SPLITS[fault]["test_original_sequences"])),
      "sampled_rows":int(len(sf)),"tier1_rows":int(sf.tier.eq("tier1").sum()),"rescue_rows":int(sf.tier.eq("rescue").sum()),
      "tier1_sequences":int(sf.loc[sf.tier.eq("tier1"),"sequence_id"].nunique()),
      "rescue_sequences":int(sf.loc[sf.tier.eq("rescue"),"sequence_id"].nunique()),
      "positive_rows":int(EVAL.loc[EVAL.fault.eq(fault),"label_near_5k"].sum())}
summary={"completed":True,"stop_condition":None,"notebook":str(NOTEBOOK_PATH),
 "git_commit":GIT_HASH,"python_environment":{"version":"3","platform":platform.platform()},
 "source_paths":{k:str(v.resolve()) for k,v in SOURCE_PATHS.items()},
 "preserved_v1_prereg_bytes_sha256":V1_BYTES_SHA256,"prereg_v2_bytes_sha256":V2_SHA256,"prereg_v2_1_bytes_sha256":PREREG_SHA256,
 "amendment_reason":PREREG["amendment_reason"],"main_MODEL_UNAVAILABLE":MAIN_UNAVAILABLE_N,"main_MODEL_UNAVAILABLE_rate":MAIN_UNAVAILABLE_RATE,
 "rankings_sha256":RANKINGS_SHA256,"04b_rule":"LSTM alarm and RF probability >= 0.5",
 "06a_exact_80_row_confusion":REPRO80_ACTUAL,"sample":{"total_rows":int(len(SAMPLE)),"rescue_rows":RESCUE_N,
   "per_fault":per_fault_summary},"confirmatory_feasibility":{"status":FEASIBILITY_STATUS,
   "fault1_fault2_rescue_positive_rows":primary_pos,"negative_rows":primary_neg,"rescue_sequences":primary_seq},
 "hypothesis_tests":HYPOTHESIS_TESTS,"effect_vs_04b":_rows_plain(EFFECT.loc[EFFECT.scope.eq("fault1+fault2")]),
 "cost_and_calls":CALL_SUMMARY,"labels_first_loaded_utc":LABELS_FIRST_LOAD_UTC,
 "limits":["Percentile sample of LSTM alarm endpoints; not all endpoints or a population event-rate estimate.",
  "fault3 test sequences were examined in the 06 series and are not completely unseen.",
  "fault1+fault2 is the primary confirmatory population; fault3 is reported separately."],
 "output_files":["prereg_07_v2.json","eval_sample.csv","eval_labels.csv","tier_split.csv","jev_R2_scores.csv",
  "llm_noRF_results.jsonl","llm_stability.csv","rankings.csv","hypothesis_tests.json","effect_vs_04b.csv",
  "cost_curve.csv","exploratory.csv","trial_log.md","summary.json"]}
_write_json(ART_DIR/"summary.json",summary)
with (ART_DIR/"trial_log.md").open("a",encoding="utf-8") as f:
    f.write(f"\n## Final results\n- actual API requests: {CALL_SUMMARY['actual_API_requests']}\n")
    f.write(f"- retries: {retry_count}; unavailable outputs: {unavailable_count}\n")
    f.write(f"- preregistered expected cost USD: {EST_TOTAL:.8f}; cap USD: {COST_CAP:.8f}\n")
    f.write(f"- actual API reported USD: {CALL_SUMMARY['07b_API_cost_reported_usd']:.8f}; fallback estimated USD: {CALL_SUMMARY['07b_API_cost_fallback_estimated_usd']:.8f}\n")
    f.write(f"- guard spend USD: {budget_snapshot['guard_spend_usd']:.8f}; rankings SHA-256: {RANKINGS_SHA256}\n")
    f.write(f"- labels first loaded UTC: {summary['labels_first_loaded_utc']}\n- stop-condition: none\n")
    f.write(f"- feasibility: {FEASIBILITY_STATUS}; H1={H1_DECISION}; H2={H2_DECISION}; H3={H3_DECISION}\n")
    f.write(f"- JEV routing LLM-equivalent calls: {len(mid_keys)}/{len(RESCUE)}; equivalent cost saving USD: {saving_usd:.8f} ({saving_pct if saving_pct is not None else 'NA'}%)\n")
assert_prereg_hash()
print("confirmatory feasibility:",FEASIBILITY_STATUS,"H1/H2/H3:",H1_DECISION,H2_DECISION,H3_DECISION)
print("H1/H2/H3 estimates:",H1_EST,H1_EST-H2_LLM,H1_EST-H3_RANDOM,"lower bounds:",H1_LOWER,H2_LOWER,H3_LOWER)
print("E20 effect rows:")
display(EFFECT.loc[EFFECT.E.eq(20)])
print("call and cost summary:",CALL_SUMMARY)
print("output files:",summary["output_files"])
def unavailable_exclusion_sensitivity():
    RESCUE=globals()["RESCUE"].loc[~globals()["RESCUE"].row_key.isin(MAIN_UNAVAILABLE_KEYS)].copy()
    keep=set(RESCUE.row_key)
    primary=EVAL.loc[EVAL.row_key.isin(keep)&EVAL.fault.isin(["fault1","fault2"])]
    primary_pos=int(primary.label_near_5k.sum()); primary_neg=len(primary)-primary_pos
    primary_seq=int(primary.sequence_id.nunique())
    FEASIBLE=primary_pos>=30 and primary_neg>=30 and primary_seq>=10
    FEASIBILITY_STATUS="PASS" if FEASIBLE else "INSUFFICIENT_CONFIRMATORY_SAMPLE"
    RANKING_BY_FAULT={m:{f:[k for k in order if k in keep] for f,order in fs.items()} for m,fs in globals()["RANKING_BY_FAULT"].items()}
    RANDOM_RANKINGS=[{**r,"ranks":{k:v for k,v in r["ranks"].items() if k in keep}} for r in globals()["RANDOM_RANKINGS"]]
    assert_prereg_hash()
    from scipy.stats import hypergeom
    from collections import Counter
    Y_MAP=dict(zip(EVAL_LABELS.row_key,EVAL_LABELS.label_near_5k.astype(int)))
    GROUP_MAP=dict(zip(RESCUE.row_key,RESCUE.sequence_group))
    FAULT_MAP=dict(zip(RESCUE.row_key,RESCUE.fault))
    JEV_TIER=route_tiers["JEV_routing"]
    RANDOM_BY_FAULT_REP={(x["fault"],x["rep"]):x["ranks"] for x in RANDOM_RANKINGS}

    def _order(method,fault,rep=None):
        if method=="random":
            rd=RANDOM_BY_FAULT_REP[(fault,int(rep))]
            return sorted(rd,key=lambda k:(rd[k],k))
        return RANKING_BY_FAULT[method][fault]

    def _top(method,fault,fraction,rep=None):
        order=_order(method,fault,rep)
        k=round_half_up_count(len(order),fraction)
        return order[:k]

    def _precision(method,faults,fraction,rep=None):
        positive=reviewed=0
        for fault in faults:
            selected=_top(method,fault,fraction,rep)
            reviewed+=len(selected); positive+=sum(Y_MAP[k] for k in selected)
        return (positive/reviewed if reviewed else float("nan")),positive,reviewed

    PRIMARY_FAULTS=("fault1","fault2")
    H1_EST,H1_POS,H1_K=_precision("JEV_routing",PRIMARY_FAULTS,.20)
    H2_LLM,H2_POS,H2_K=_precision("full_LLM",PRIMARY_FAULTS,.20)
    PRIMARY_BASE=primary_pos/len(primary)
    random_precision_by_rep=[]
    for rep in range(1000):
        p,pos,k=_precision("random",PRIMARY_FAULTS,.20,rep)
        random_precision_by_rep.append(p)
    H3_RANDOM=float(np.mean(random_precision_by_rep))

    # Sequence-cluster bootstrap. Duplicate selected sequences with all sampled rows;
    # the ranking order itself remains frozen and label-blind.
    BOOT_REPS=10000
    BOOT_DISTRIBUTIONS={"H1":[],"H2":[],"H3":[]}
    if FEASIBLE and FORMAL_ALLOWED:
        group_keys={f:sorted(RESCUE.loc[RESCUE.fault.eq(f),"sequence_group"].unique().tolist()) for f in PRIMARY_FAULTS}
        rows_by_fault={f:RESCUE.loc[RESCUE.fault.eq(f),"row_key"].tolist() for f in PRIMARY_FAULTS}
        j_order={f:RANKING_BY_FAULT["JEV_routing"][f] for f in PRIMARY_FAULTS}
        l_order={f:RANKING_BY_FAULT["full_LLM"][f] for f in PRIMARY_FAULTS}
        boot_rng=np.random.default_rng(RANDOM_SEED)
        for boot_i in range(BOOT_REPS):
            pooled_j_pos=pooled_j_k=pooled_l_pos=pooled_l_k=0
            pooled_y=pooled_n=pooled_rand_exp=pooled_rand_k=0.0
            for fault in PRIMARY_FAULTS:
                groups=group_keys[fault]
                sampled_groups=boot_rng.choice(np.asarray(groups,dtype=object),size=len(groups),replace=True).tolist()
                mult=Counter(sampled_groups)
                def expand(order):
                    return [key for key in order for _ in range(mult.get(GROUP_MAP[key],0))]
                ej=expand(j_order[fault]); el=expand(l_order[fault]); all_rows=expand(rows_by_fault[fault])
                n=len(all_rows); k=round_half_up_count(n,.20)
                yp=sum(Y_MAP[x] for x in all_rows); prevalence=yp/n if n else 0.0
                pooled_y+=yp; pooled_n+=n
                pooled_j_pos+=sum(Y_MAP[x] for x in ej[:k]); pooled_j_k+=min(k,len(ej))
                pooled_l_pos+=sum(Y_MAP[x] for x in el[:k]); pooled_l_k+=min(k,len(el))
                b=sum(mult.get(GROUP_MAP[key],0) for key in rows_by_fault[fault] if JEV_TIER.get(key)=="MID")
                h=round_half_up_count(n-b,.50)
                if k<=h:
                    exp_random=k*prevalence
                else:
                    q_mid=min(max(0,k-h),b)
                    exp_random=h*prevalence
                    if q_mid and b:
                        # Uniform random MID subset; within it, the frozen LLM ordering is used.
                        positions=np.arange(n,dtype=int)
                        probs=(b/n)*hypergeom.cdf(q_mid-1,n-1,positions,b-1)
                        y_llm=np.asarray([Y_MAP[x] for x in el],dtype=float)
                        exp_random+=float(np.dot(probs,y_llm))
                    low_take=max(0,k-h-b)
                    if low_take: exp_random+=low_take*prevalence
                pooled_rand_exp+=exp_random; pooled_rand_k+=k
            jp=pooled_j_pos/pooled_j_k if pooled_j_k else 0.0
            lp=pooled_l_pos/pooled_l_k if pooled_l_k else 0.0
            base=pooled_y/pooled_n if pooled_n else 0.0
            rp=pooled_rand_exp/pooled_rand_k if pooled_rand_k else 0.0
            BOOT_DISTRIBUTIONS["H1"].append(jp-base)
            BOOT_DISTRIBUTIONS["H2"].append(jp-lp)
            BOOT_DISTRIBUTIONS["H3"].append(jp-rp)

    def _lower(values): return float(np.quantile(np.asarray(values,dtype=float),.05)) if values else None
    H1_LOWER=_lower(BOOT_DISTRIBUTIONS["H1"]); H2_LOWER=None; H3_LOWER=None
    if not FORMAL_ALLOWED:
        H1_DECISION=H2_DECISION=H3_DECISION="DESCRIPTIVE_ONLY_MODEL_UNAVAILABLE"
        H1_GATE=H2_GATE=H3_GATE="More than 5% main rescue LLM unavailable; no formal testing"
    elif not FEASIBLE:
        H1_DECISION=H2_DECISION=H3_DECISION="INSUFFICIENT_CONFIRMATORY_SAMPLE"
        H1_GATE=H2_GATE=H3_GATE="sample feasibility condition not met; formal test not run"
    else:
        H1_DECISION="PASS" if H1_LOWER>0 else "FAIL_TO_REJECT"
        H1_GATE="TESTED"
        if H1_DECISION!="PASS":
            H2_DECISION=H3_DECISION="NOT_TESTED_DUE_TO_GATEKEEPING"
            H2_GATE=H3_GATE="H1 did not pass"
        else:
            H2_LOWER=_lower(BOOT_DISTRIBUTIONS["H2"])
            H2_DECISION="PASS" if H2_LOWER>-.05 else "FAIL_TO_REJECT"
            H2_GATE="TESTED"
            if H2_DECISION!="PASS":
                H3_DECISION="NOT_TESTED_DUE_TO_GATEKEEPING"; H3_GATE="H2 did not pass"
            else:
                H3_LOWER=_lower(BOOT_DISTRIBUTIONS["H3"])
                H3_DECISION="PASS" if H3_LOWER>0 else "FAIL_TO_REJECT"; H3_GATE="TESTED"
    HYPOTHESIS_TESTS={
     "primary_population":"fault1+fault2 combined rescue","primary_budget":"E20 per fault, half-up",
     "alpha":.05,"bootstrap_repetitions":BOOT_REPS,"bootstrap_seed":RANDOM_SEED,"random_routing_seed":RANDOM_SEED+1,
     "bootstrap_unit":"sequence within fault; all rescue rows retained per sampled sequence",
     "feasibility":{"positive_rows":primary_pos,"negative_rows":primary_neg,"rescue_sequences":primary_seq,
                     "status":FEASIBILITY_STATUS},
     "H1":{"hypothesis":"JEV precision@E20 - rescue prevalence > 0","population":"fault1+fault2 rescue",
      "budget":"E20","estimate":float(H1_EST),"comparator_estimate":float(PRIMARY_BASE),
      "difference":float(H1_EST-PRIMARY_BASE),"bootstrap_repetitions":BOOT_REPS if FEASIBLE and FORMAL_ALLOWED else 0,
      "alpha":.05,"one_sided_lower_bound":H1_LOWER,"threshold":0.0,"decision":H1_DECISION,
      "testing_order":1,"gatekeeping_status":H1_GATE},
     "H2":{"hypothesis":"JEV precision@E20 - full LLM precision@E20 > -0.05","population":"fault1+fault2 rescue",
      "budget":"E20","estimate":float(H1_EST),"comparator_estimate":float(H2_LLM),
      "difference":float(H1_EST-H2_LLM),"bootstrap_repetitions":BOOT_REPS if FEASIBLE and H2_GATE=="TESTED" else 0,
      "alpha":.05,"one_sided_lower_bound":H2_LOWER,"threshold":-.05,"noninferiority_margin":-.05,
      "decision":H2_DECISION,"testing_order":2,"gatekeeping_status":H2_GATE},
     "H3":{"hypothesis":"JEV precision@E20 - mean random routing precision@E20 > 0","population":"fault1+fault2 rescue",
      "budget":"E20","estimate":float(H1_EST),"comparator_estimate":float(H3_RANDOM),
      "difference":float(H1_EST-H3_RANDOM),"bootstrap_repetitions":BOOT_REPS if FEASIBLE and H3_GATE=="TESTED" else 0,
      "alpha":.05,"one_sided_lower_bound":H3_LOWER,"threshold":0.0,"decision":H3_DECISION,
      "testing_order":3,"gatekeeping_status":H3_GATE,
      "random_routing_repetitions":1000,"bootstrap_random_comparator":"exact conditional expectation of preregistered randomized HIGH/MID/LOW route; point estimate is mean of 1,000 seeded routes"}
    }
    HYPOTHESIS_TESTS["analysis_type"]="Sensitivity only; does not change primary decisions"
    HYPOTHESIS_TESTS["excluded_main_MODEL_UNAVAILABLE"]=MAIN_UNAVAILABLE_N
    return HYPOTHESIS_TESTS
SENSITIVITY=unavailable_exclusion_sensitivity()
_write_json(ART_DIR/"sensitivity_excluding_MODEL_UNAVAILABLE.json",SENSITIVITY)
summary["sensitivity_excluding_MODEL_UNAVAILABLE"]=SENSITIVITY
summary["output_files"].extend(["prereg_07_v2_1.json","sensitivity_excluding_MODEL_UNAVAILABLE.json","actual_request_body_example.json","availability_diagnostic.json","first20_diagnostic.json","conclusion.md"])
_write_json(ART_DIR/"summary.json",summary)
_append_log(f"- v2.1 amendment: {PREREG['amendment_reason']}\n- main MODEL_UNAVAILABLE: {MAIN_UNAVAILABLE_N}/{RESCUE_N}\n- sensitivity H1/H2/H3: {[SENSITIVITY[h]['decision'] for h in ('H1','H2','H3')]}\n")


## 9. Notebook conclusion for the 08 final report

In [ ]:
def _pct(x): return "N/A" if x is None or not np.isfinite(float(x)) else f"{100*float(x):.1f}%"
def _hline(name,decision,diff,lower,threshold):
    if decision=="DESCRIPTIVE_ONLY_MODEL_UNAVAILABLE": return f"- {name}: MODEL_UNAVAILABLE 비율이 5%를 초과해 기술 통계만 보고합니다."
    if decision=="INSUFFICIENT_CONFIRMATORY_SAMPLE": return f"- {name}: 표본 조건 미달(`INSUFFICIENT_CONFIRMATORY_SAMPLE`); formal 판정은 하지 않았습니다."
    if decision=="NOT_TESTED_DUE_TO_GATEKEEPING": return f"- {name}: gatekeeping 때문에 검정하지 않았습니다(`NOT_TESTED_DUE_TO_GATEKEEPING`)."
    return f"- {name}: {decision}; 차이={float(diff):.3f}, one-sided 95% lower bound={float(lower):.3f}, 기준>{threshold}."
primary_e20=EFFECT.loc[(EFFECT.scope=="fault1+fault2")&(EFFECT.E==20)].iloc[0]
e10=EFFECT.loc[(EFFECT.scope=="fault1+fault2")&(EFFECT.E==10)].iloc[0]
e30=EFFECT.loc[(EFFECT.scope=="fault1+fault2")&(EFFECT.E==30)].iloc[0]
f3_e20=EFFECT.loc[(EFFECT.scope=="fault3")&(EFFECT.E==20)].iloc[0]
conclusion=f"""### v2.1 개정 및 실패 처리

라벨 개봉과 순위 확정 전에 v2.1로 개정했습니다: {PREREG['amendment_reason']}
요청의 strict json_schema는 이미 적용되어 설정을 유지했습니다. 실패 시 동일 요청을 최대 2회 재시도했습니다.
Main rescue MODEL_UNAVAILABLE은 **{MAIN_UNAVAILABLE_N}/{RESCUE_N}건 ({MAIN_UNAVAILABLE_RATE:.1%})**이며, 안정성 호출에서는 **{int(STABILITY.status.ne('success').sum())}/15건**입니다. v2의 실패 원문은 당시 저장되지 않아 복구할 수 없고, v2.1 실패 원문은 failed_raw에 기록했습니다.

### 확증 결과

{_hline('H1',H1_DECISION,H1_EST-PRIMARY_BASE,H1_LOWER,0)}
{_hline('H2',H2_DECISION,H1_EST-H2_LLM,H2_LOWER,-.05)}
{_hline('H3',H3_DECISION,H1_EST-H3_RANDOM,H3_LOWER,0)}

확증 표본 조건: fault1+2 rescue positive {primary_pos}, negative {primary_neg}, sequences {primary_seq} — **{FEASIBILITY_STATUS}**. Primary 분석은 fault1+2의 rescue만 대상으로 하며 tier1은 H1 precision의 분모에 넣지 않았습니다.

### 04b 대비 효과 크기

E20에서 rescue {int(primary_e20.additional_review_rows)}행을 추가 검토해 {int(primary_e20.imminent_failures_recovered)}건을 회수했습니다. rescue recovery rate는 {_pct(primary_e20.rescue_recovery_rate)}이며, sampled 전체 positives 기준 recall은 04b {_pct(primary_e20['04b_recall'])}에서 구조 lane 추가 후 {_pct(primary_e20.structural_recall)}로 바뀌었습니다({_pct(primary_e20.recall_change_vs_04b)}p 증가). 보조 E10은 {int(e10.imminent_failures_recovered)}/{int(e10.additional_review_rows)} 회수, E30은 {int(e30.imminent_failures_recovered)}/{int(e30.additional_review_rows)} 회수입니다. fault3 E20은 별도 보고로 {int(f3_e20.imminent_failures_recovered)}/{int(f3_e20.additional_review_rows)} 회수했습니다.

### LLM 호출 절감

Full LLM 기준 {len(RESCUE)} rescue 행에 대해 호출했고, JEV routing의 실제 정수 MID 예산은 {len(mid_keys)}행({llm_reduction_pct:.1f}% fewer LLM-equivalent calls)입니다. 현 실험의 실제 API requests는 JEV {CALL_COUNTS['jev']}, main LLM {CALL_COUNTS['llm_main']}, stability LLM {CALL_COUNTS['llm_stability']}회입니다. 응답 usage cost와 필요한 경우 07a 단가 fallback 기준으로 full LLM equivalent cost ${full_llm_equiv_cost:.6f}, JEV+MID LLM scenario cost ${routed_total_equiv_cost:.6f}, 추정 절감 ${saving_usd:.6f} ({saving_pct if saving_pct is not None else 'N/A'}%)입니다. 실제 07b 실행 API spend는 reported ${CALL_SUMMARY['07b_API_cost_reported_usd']:.6f} + fallback estimate ${CALL_SUMMARY['07b_API_cost_fallback_estimated_usd']:.6f}입니다.

### MODEL_UNAVAILABLE 제외 민감도 분석

주 분석의 고정 순위에서 실패 행을 제외하고 E20·prevalence·sequence bootstrap을 다시 계산했습니다. 아래 결과는 민감도 분석이며 최종 판정에는 주 분석만 사용합니다.

{chr(10).join(f"- {h}: 차이={SENSITIVITY[h]['difference']:.4f}, 하한={SENSITIVITY[h]['one_sided_lower_bound']}, 상태={SENSITIVITY[h]['decision']}" for h in ('H1','H2','H3'))}

### 탐색 결과

아래 routing 비교, logistic/LSTM-only 비교, E10/E30, 개별 fault 및 LLM stability 결과는 모두 **탐색**이며 추가 p-value나 formal hypothesis로 해석하지 않습니다. 결과는 `exploratory.csv`에 있습니다.

### 한계

- 평가 표본은 총 {len(SAMPLE)}행이며 fault1+2 primary rescue {len(primary)}행입니다. 표본 조건 상태는 **{FEASIBILITY_STATUS}**입니다.
- 표본은 각 test sequence의 LSTM alarm endpoint에서 고정 percentile로 뽑았습니다. 모든 alarm endpoint나 운영 전체의 event rate를 대표하지는 않습니다.
- fault3 test는 06 series에서 이미 확인한 이력이 있어 완전히 unseen인 확증 세트가 아닙니다.
- primary confirmatory 집단은 fault1+2이며, fault3 결과는 별도로 보고합니다. 결과는 이 표본/프로토콜 밖으로 일반화하지 않습니다.

### 08 최종 보고용 고정 수치

- 04b/06a regression: TP={REPRO80_ACTUAL['TP']}, FP={REPRO80_ACTUAL['FP']}, FN={REPRO80_ACTUAL['FN']}, TN={REPRO80_ACTUAL['TN']}.
- JEV routing E20 precision={H1_EST:.3f}; rescue prevalence={PRIMARY_BASE:.3f}; full LLM E20 precision={H2_LLM:.3f}; random routing mean={H3_RANDOM:.3f}.
- H1/H2/H3: {H1_DECISION} / {H2_DECISION} / {H3_DECISION}; prereg SHA-256 `{PREREG_SHA256}`; rankings SHA-256 `{RANKINGS_SHA256}`.
- Stop condition: 없음. 07c는 생성하지 않았습니다.
"""
display(Markdown(conclusion))
(ART_DIR/"conclusion.md").write_text(conclusion,encoding="utf-8")